# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 10/12** (CPU, internet off). **1777 series**, ~10.6 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part10.zip`
- `manifest_rgb25d_320_part10.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 10
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231","series":"1.2.826.0.1.3680043.8.498.26515132645256697953312631089055309042","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542","series":"1.2.826.0.1.3680043.8.498.87738065849001552362672384225773846709","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101","series":"1.2.826.0.1.3680043.8.498.69783908410864161211158583420543900194","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559","series":"1.2.826.0.1.3680043.8.498.58180411352017518670923060837962824092","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.10138400860303228304047985771421435784","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018","series":"1.2.826.0.1.3680043.8.498.64945894485172824370200401071778190389","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401","series":"1.2.826.0.1.3680043.8.498.76206109519289930878461038242097331022","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922","series":"1.2.826.0.1.3680043.8.498.10615181069597744316725609035737538256","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.70860619701802043401860561907550358128","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347","series":"1.2.826.0.1.3680043.8.498.10238197199619807565017429401905996014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097","series":"1.2.826.0.1.3680043.8.498.24753705365555249891287619579982307015","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149","series":"1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364","series":"1.2.826.0.1.3680043.8.498.36057579978771860657165441134042213424","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636","series":"1.2.826.0.1.3680043.8.498.85751867415695932706674441547035128463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882","series":"1.2.826.0.1.3680043.8.498.80625470355868527215277354248817543532","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834","series":"1.2.826.0.1.3680043.8.498.94557540944998494802223205206487031009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586","series":"1.2.826.0.1.3680043.8.498.19404052662797148007645917722916874242","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471","series":"1.2.826.0.1.3680043.8.498.89907148560264215007915118880911884303","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182","series":"1.2.826.0.1.3680043.8.498.51084084454936728856990915694601621680","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824","series":"1.2.826.0.1.3680043.8.498.34551518955124776381391941770845659774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221","series":"1.2.826.0.1.3680043.8.498.59911622537708982351901979512701900871","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790","series":"1.2.826.0.1.3680043.8.498.13428571725466320588810484325028997911","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930","series":"1.2.826.0.1.3680043.8.498.12048402687794141013818869563273436477","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161","series":"1.2.826.0.1.3680043.8.498.67781858327660713056877114376962129508","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415","series":"1.2.826.0.1.3680043.8.498.59572626060453046450277523987138455921","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973","series":"1.2.826.0.1.3680043.8.498.45598525177725299928996498825173854869","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197","series":"1.2.826.0.1.3680043.8.498.10629255255006269628715273944862669667","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364","series":"1.2.826.0.1.3680043.8.498.35320024245044005696259306061920440242","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10197565004992937722841731691775578390","series":"1.2.826.0.1.3680043.8.498.20932205074673641784917157989392799178","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801","series":"1.2.826.0.1.3680043.8.498.48950480256477480101177877164709371717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516","series":"1.2.826.0.1.3680043.8.498.11067199620290436437594977824702232829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062","series":"1.2.826.0.1.3680043.8.498.29590264382520318647432972951156289862","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821","series":"1.2.826.0.1.3680043.8.498.12326169746050258981170339816434754337","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576","series":"1.2.826.0.1.3680043.8.498.11053320715247700506396081651611786054","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","series":"1.2.826.0.1.3680043.8.498.12283760903791384028030758594483070789","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833","series":"1.2.826.0.1.3680043.8.498.11211335876233026549969879030404416523","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286","series":"1.2.826.0.1.3680043.8.498.60762249674220813439837868770211697775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873","series":"1.2.826.0.1.3680043.8.498.92705138135870641016202486853104333527","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.34769620643166254696253074754381740184","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542","series":"1.2.826.0.1.3680043.8.498.50250788232183504450267442000416776519","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382","series":"1.2.826.0.1.3680043.8.498.10281359408836745211721920919607939009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10284446538566255572004044331831825072","series":"1.2.826.0.1.3680043.8.498.32830202036193166375244104661509838562","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808","series":"1.2.826.0.1.3680043.8.498.28093605160731152580137405583288916835","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106","series":"1.2.826.0.1.3680043.8.498.10423785329270878355290992751351515671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10310831394568337664174691281092566272","series":"1.2.826.0.1.3680043.8.498.37169981188815818697642650154430705725","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10314240078685804110739316196999571466","series":"1.2.826.0.1.3680043.8.498.71146215660561666216021174663885312559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548","series":"1.2.826.0.1.3680043.8.498.80469365269678833040899269297601067921","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315","series":"1.2.826.0.1.3680043.8.498.38301154405113720756877683395927348934","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332","series":"1.2.826.0.1.3680043.8.498.27973777205285389608889350303684355143","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726","series":"1.2.826.0.1.3680043.8.498.71516195382954494276876784345617287161","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936","series":"1.2.826.0.1.3680043.8.498.52505020521115481524991285675872814604","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10351168527688244230248760520733020605","series":"1.2.826.0.1.3680043.8.498.37243230079513932589115488417034538512","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197","series":"1.2.826.0.1.3680043.8.498.13256289021483697369566985758579036918","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10361030765327974510262894062047216433","series":"1.2.826.0.1.3680043.8.498.40398630019238268382816987037625346997","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218","series":"1.2.826.0.1.3680043.8.498.91810356509084694315612116355262452738","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801","series":"1.2.826.0.1.3680043.8.498.55295113882838479155887138900637714353","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341","series":"1.2.826.0.1.3680043.8.498.19437245024810777572794008127521176449","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419","series":"1.2.826.0.1.3680043.8.498.11229257950923195079166682461014606368","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007","series":"1.2.826.0.1.3680043.8.498.11492388527430571863466826301899559225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409","series":"1.2.826.0.1.3680043.8.498.45796816978981037426211882076017117096","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713","series":"1.2.826.0.1.3680043.8.498.13185080101639073577674981654606412299","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10419958821528108176389641942168280311","series":"1.2.826.0.1.3680043.8.498.28431176017159629090584749476525161560","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689","series":"1.2.826.0.1.3680043.8.498.75429038960118708183254672641088728856","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035","series":"1.2.826.0.1.3680043.8.498.34459191379528356295889111188107065683","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805","series":"1.2.826.0.1.3680043.8.498.12023140751484285734081763729417321305","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.49376918330415107224934671356580803207","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509","series":"1.2.826.0.1.3680043.8.498.12557256100879734517965569754969876670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674","series":"1.2.826.0.1.3680043.8.498.97639855332563872908207073072676858618","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884","series":"1.2.826.0.1.3680043.8.498.10147116190126634468469610209040489806","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091","series":"1.2.826.0.1.3680043.8.498.79689803747379857208090951595847632334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625","series":"1.2.826.0.1.3680043.8.498.54362742633933893689205396401535453547","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.12066111103599414456618020249053783516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078","series":"1.2.826.0.1.3680043.8.498.10069091463980274550818866947208216218","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10511712617129216774539441890488742560","series":"1.2.826.0.1.3680043.8.498.20606053226335793166211893097454142913","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627","series":"1.2.826.0.1.3680043.8.498.81870892204470390922627513929212004813","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196","series":"1.2.826.0.1.3680043.8.498.77566849991640864833457772373049521791","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10532708800468111850607752497432656694","series":"1.2.826.0.1.3680043.8.498.81450284352955030269204147369491604197","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10536557774352716497356905292652597953","series":"1.2.826.0.1.3680043.8.498.22190263446184527904190559033613858671","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828","series":"1.2.826.0.1.3680043.8.498.85285866926488618424127672418510763643","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.22581536122871440772283941981232719911","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216","series":"1.2.826.0.1.3680043.8.498.11448894224088044090557083113083709825","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953","series":"1.2.826.0.1.3680043.8.498.79194086061555794406311099605816715714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260","series":"1.2.826.0.1.3680043.8.498.21570196298309210291885243085764510163","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084","series":"1.2.826.0.1.3680043.8.498.63812132595953551557962684818043251692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10587511485561679190576188107774736852","series":"1.2.826.0.1.3680043.8.498.39242858921693719086588147102159354048","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10590169032156570386146081157598789893","series":"1.2.826.0.1.3680043.8.498.17747241487778230931717092731509816573","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628","series":"1.2.826.0.1.3680043.8.498.73208530462222804085123162930370132010","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146","series":"1.2.826.0.1.3680043.8.498.13185105048054281108510659492260791806","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196","series":"1.2.826.0.1.3680043.8.498.98164835603842188983536557989473362651","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514","series":"1.2.826.0.1.3680043.8.498.12525105533574085185164734710480228348","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10619486897279334206912764495067756449","series":"1.2.826.0.1.3680043.8.498.50176703731215284144470430416209475585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593","series":"1.2.826.0.1.3680043.8.498.35943380381764774388878867287077179456","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971","series":"1.2.826.0.1.3680043.8.498.51160291043259015085201545280113711796","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10632177122947017620482277062226288392","series":"1.2.826.0.1.3680043.8.498.63258260445214921314382295711113660702","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10648346793256473462304051626410900340","series":"1.2.826.0.1.3680043.8.498.62233238841828328533125303158888776931","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10652685418462611085839874321267337079","series":"1.2.826.0.1.3680043.8.498.79982534358896588607326098685753566714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996","series":"1.2.826.0.1.3680043.8.498.37844149882135778138833598168193560518","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356","series":"1.2.826.0.1.3680043.8.498.89281222299822380540083609011822279163","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040","series":"1.2.826.0.1.3680043.8.498.78218950762632185469646293642065526827","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331","series":"1.2.826.0.1.3680043.8.498.11374922632900958449576641474355680972","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071","series":"1.2.826.0.1.3680043.8.498.11977145875178265168440720275556462253","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286","series":"1.2.826.0.1.3680043.8.498.10819641419973377990103001007748128579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.14311318974303035256807030420716595509","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10715956446507906169485719013515169244","series":"1.2.826.0.1.3680043.8.498.24934072323257177384038879562126606253","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10728448245470003618625208276233778895","series":"1.2.826.0.1.3680043.8.498.28308037197371673323661359196733765522","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171","series":"1.2.826.0.1.3680043.8.498.13152174648534196023734086893409472693","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.93430662806389225278588343422834573092","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375","series":"1.2.826.0.1.3680043.8.498.94660716470173139710060179213731711084","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10751624265599812767108610284996090628","series":"1.2.826.0.1.3680043.8.498.87721781263984814967279237285574891842","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054","series":"1.2.826.0.1.3680043.8.498.42740877287860617670311897851593817502","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610","series":"1.2.826.0.1.3680043.8.498.88089865919457051504930941581723273203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428","series":"1.2.826.0.1.3680043.8.498.11006910962228708750296692735183177399","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10770877369291591553624215762368014357","series":"1.2.826.0.1.3680043.8.498.28078018445154230067976395785693560154","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334","series":"1.2.826.0.1.3680043.8.498.38584589547138755790598179288445410978","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10802653832050736637820225444196833450","series":"1.2.826.0.1.3680043.8.498.93126934310826426351524679908174850904","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588","series":"1.2.826.0.1.3680043.8.498.63420876273169778546266694899637021395","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528","series":"1.2.826.0.1.3680043.8.498.32313617084861986764034097455265452407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306","series":"1.2.826.0.1.3680043.8.498.11881499706488230273674690075693732717","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10839126901366636822676830560774505865","series":"1.2.826.0.1.3680043.8.498.45708598993264803573170681224796030432","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972","series":"1.2.826.0.1.3680043.8.498.65790813120751233108127728082262539218","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10850742579000782417673172155504794301","series":"1.2.826.0.1.3680043.8.498.85269246959179815285767515178965737876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10854012268828835845581966634225522931","series":"1.2.826.0.1.3680043.8.498.18806883525677875115670223886191096006","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655","series":"1.2.826.0.1.3680043.8.498.74059629671809506861861345318428729944","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10862481204237376580557516731952961213","series":"1.2.826.0.1.3680043.8.498.17035273029093724180460733956975820599","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617","series":"1.2.826.0.1.3680043.8.498.92463453066307270178961393649259400682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10886583574130697346788286647281863932","series":"1.2.826.0.1.3680043.8.498.34094228373563356071448146139348611490","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673","series":"1.2.826.0.1.3680043.8.498.57276679883671168368261891732126995443","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10917389623891555306584069980815338466","series":"1.2.826.0.1.3680043.8.498.74126342625320754630904391710961526722","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433","series":"1.2.826.0.1.3680043.8.498.28970342926907545782381263795321818683","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10924685588962763768106617368647678902","series":"1.2.826.0.1.3680043.8.498.73659798219516555920916239976715773129","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.67070623886557207951623419646045810532","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737","series":"1.2.826.0.1.3680043.8.498.49042962527541240465600822355201725571","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722","series":"1.2.826.0.1.3680043.8.498.23065195904724610449900822761164486278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.19203615667587645395484665964089253197","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670","series":"1.2.826.0.1.3680043.8.498.74185874352645956314216062499329443281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575","series":"1.2.826.0.1.3680043.8.498.83144091158629302282978436418775858977","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415","series":"1.2.826.0.1.3680043.8.498.67235052812284911073248085414863632772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424","series":"1.2.826.0.1.3680043.8.498.11426302942225368123189570593091708388","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699","series":"1.2.826.0.1.3680043.8.498.17917547375209705407372819100665273333","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174","series":"1.2.826.0.1.3680043.8.498.66486664749399369616211426422442358162","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480","series":"1.2.826.0.1.3680043.8.498.27491465685211711764956561078877405918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11006241238894862275805535376070111510","series":"1.2.826.0.1.3680043.8.498.13606840504138114964136479712907945500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11010004366885474301784150582039137111","series":"1.2.826.0.1.3680043.8.498.38581764120546188217187599355004256223","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310","series":"1.2.826.0.1.3680043.8.498.72869894823761377178626980069986100075","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11043081829401721095803441219553837608","series":"1.2.826.0.1.3680043.8.498.15575148791689906749155515639520370018","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545","series":"1.2.826.0.1.3680043.8.498.91958884386719681176898542001198366358","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008","series":"1.2.826.0.1.3680043.8.498.12069770320340355266813912499570590170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11057883757434391643878168168845592475","series":"1.2.826.0.1.3680043.8.498.31156808966407755378141371966857780534","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11067200656721609390997819562401839124","series":"1.2.826.0.1.3680043.8.498.35710470825393408313868442377361016186","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995","series":"1.2.826.0.1.3680043.8.498.32704514397768378725363641942016415579","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11084010833221855396433326769153950681","series":"1.2.826.0.1.3680043.8.498.83992799685739199655757662129593433472","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233","series":"1.2.826.0.1.3680043.8.498.16390565461010258635781204255447989836","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11110575656494405149601490027444484170","series":"1.2.826.0.1.3680043.8.498.12262836427153959933759180750019030550","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11113442398569287228475456000446876768","series":"1.2.826.0.1.3680043.8.498.21599274942516200255395086865270640569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.19718225929787549754525725052233684236","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11126383719852491587081347886000714259","series":"1.2.826.0.1.3680043.8.498.56277946662012745845533578136442622874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11131210140372132238166068091728877219","series":"1.2.826.0.1.3680043.8.498.10569134517500088866045246021220344474","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286","series":"1.2.826.0.1.3680043.8.498.72058290790207727572875316450742118448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11148939654362681630811787929668902046","series":"1.2.826.0.1.3680043.8.498.10041562232928602524283287007509887763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150193626240150189232364321568347937","series":"1.2.826.0.1.3680043.8.498.68792888495812990470387546601507648880","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033","series":"1.2.826.0.1.3680043.8.498.85410269922026313487385431623624119038","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11156884483511690263238401598901614096","series":"1.2.826.0.1.3680043.8.498.12521492305783891361169528265104161535","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11180457573346204459956051956213528997","series":"1.2.826.0.1.3680043.8.498.12926903548458857372432280452782156734","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747","series":"1.2.826.0.1.3680043.8.498.10747058567857137959502943890477960088","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11191015483427903713800997920940844550","series":"1.2.826.0.1.3680043.8.498.30980863675935941008105037879776019329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878","series":"1.2.826.0.1.3680043.8.498.41722585911105554899014594572153154832","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.10401203504595896441928605534503423062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11211598263863419448236436640277320409","series":"1.2.826.0.1.3680043.8.498.75369379509711661645306257881636666624","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892","series":"1.2.826.0.1.3680043.8.498.11395693125165982491594775518682316155","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11219813299300915347039206446596901205","series":"1.2.826.0.1.3680043.8.498.13969035339623676527610657441782058619","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884","series":"1.2.826.0.1.3680043.8.498.10473540547191433687694601151903395338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11231822445970527276203191047278744341","series":"1.2.826.0.1.3680043.8.498.32427460475862439835083535246920188220","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11244115315611630362929576340631670750","series":"1.2.826.0.1.3680043.8.498.32492712405425916078889107085276910765","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11257168136762043265165109052826442974","series":"1.2.826.0.1.3680043.8.498.82587999721527403817522028413484578864","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683","series":"1.2.826.0.1.3680043.8.498.96755556085603353431682778965644039754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204","series":"1.2.826.0.1.3680043.8.498.55684447796439065534126985930419373907","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11275711698468745807758222802302685022","series":"1.2.826.0.1.3680043.8.498.26316154085257660944362140295921171734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291","series":"1.2.826.0.1.3680043.8.498.32013233467326527147343878483254663222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11282162920338602001456704018617788412","series":"1.2.826.0.1.3680043.8.498.10096877589530261930201157322108976095","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580","series":"1.2.826.0.1.3680043.8.498.12552728774210526013400534363105740572","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11294885816514924203337081832122619605","series":"1.2.826.0.1.3680043.8.498.10956577345697700185152822154174533511","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11296867854260286881418279496022384461","series":"1.2.826.0.1.3680043.8.498.35670558401211449970309107264580233401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411","series":"1.2.826.0.1.3680043.8.498.32842610905704128813962705024290938448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.12977540772282680944351030635169252418","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11316091466685605758061778045311969672","series":"1.2.826.0.1.3680043.8.498.76273946509194426938485168880873571658","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11334776960319816844935217444817103072","series":"1.2.826.0.1.3680043.8.498.10258335628440626484043752536414887798","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260","series":"1.2.826.0.1.3680043.8.498.92419831488128309692880819094800061194","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11347466940974045182979684641300336177","series":"1.2.826.0.1.3680043.8.498.49052384713593462925981483279472545822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11362547519125634779422991321337283241","series":"1.2.826.0.1.3680043.8.498.20296656526029771638976556944762029516","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11371400503360468962117579439852064607","series":"1.2.826.0.1.3680043.8.498.71824218488830583118310862947161725016","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994","series":"1.2.826.0.1.3680043.8.498.58045975927600155673486290891413127957","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11391095418955817124325938917797687542","series":"1.2.826.0.1.3680043.8.498.84631574742267159099180444272229944285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849","series":"1.2.826.0.1.3680043.8.498.78835425419711433570970596552246722117","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11406381777983128446583783158882827226","series":"1.2.826.0.1.3680043.8.498.46623836859261673394535982928177749080","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11420670844338686032030860787688858057","series":"1.2.826.0.1.3680043.8.498.63214975678437630838730885123489533361","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11425683153901568541658945026861110110","series":"1.2.826.0.1.3680043.8.498.59650893083756510861932323391126020428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11432815311029789072253610823554308183","series":"1.2.826.0.1.3680043.8.498.45734673806766715847001643346989740014","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11441489276425334980205891153179893797","series":"1.2.826.0.1.3680043.8.498.12758420266157558709171834059948558796","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579","series":"1.2.826.0.1.3680043.8.498.34458041690717786320883335753920466232","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11451981256762378203455481812714370193","series":"1.2.826.0.1.3680043.8.498.11234403202012513118931312482178349004","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11458488293461896930750018852127694875","series":"1.2.826.0.1.3680043.8.498.24419642536283649015078128154980759688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11465586454035959390661499120751232686","series":"1.2.826.0.1.3680043.8.498.32870664531468346700783377850706213790","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047","series":"1.2.826.0.1.3680043.8.498.39670754130438950973865394745200825625","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11477000011453869751231224714283666904","series":"1.2.826.0.1.3680043.8.498.13244802252968084305482488776995607280","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719","series":"1.2.826.0.1.3680043.8.498.12072873689223532611438474852441119647","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434","series":"1.2.826.0.1.3680043.8.498.12353550914676654629968567283367475989","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11498239309891927151456629915927889165","series":"1.2.826.0.1.3680043.8.498.12014881867631165354703273230911409710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11501070108039994411346060488666341726","series":"1.2.826.0.1.3680043.8.498.50385292659884231513896528081929364705","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11506642732332295874076832453198901115","series":"1.2.826.0.1.3680043.8.498.23012285169187003454581778719951147121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664","series":"1.2.826.0.1.3680043.8.498.49771426446550626043206738713299398504","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11516842786271998149681118182571061813","series":"1.2.826.0.1.3680043.8.498.90845824898736053419081428374690591361","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388","series":"1.2.826.0.1.3680043.8.498.86714564965204314396244030975858209238","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11532517984322509215429681648257299355","series":"1.2.826.0.1.3680043.8.498.10104470825042832140370495772349619437","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11543622448213658856759993374276266341","series":"1.2.826.0.1.3680043.8.498.50287164517847509056183414362991133929","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11547192037231064935759879673500408149","series":"1.2.826.0.1.3680043.8.498.15910929636581838565036637462302679152","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11548839502674800051973767797346682106","series":"1.2.826.0.1.3680043.8.498.68610375922699818831444330091653967655","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840","series":"1.2.826.0.1.3680043.8.498.81787334059184556686818388901104015308","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11569355546217846610131708883985793793","series":"1.2.826.0.1.3680043.8.498.16614534580281116483739727827399534829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11574480940878399452466699512074769686","series":"1.2.826.0.1.3680043.8.498.13065024032097247888486770627017173542","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804","series":"1.2.826.0.1.3680043.8.498.74879242697559101331080085342796056054","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.20627850319982740263289394922206621366","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193","series":"1.2.826.0.1.3680043.8.498.67599935957622051533829293650249121345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11592838054859712596515534341207971997","series":"1.2.826.0.1.3680043.8.498.10545601881324341812547497155772832605","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584","series":"1.2.826.0.1.3680043.8.498.54905273408354012316096050699797114264","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11599387556469808488160280438305325860","series":"1.2.826.0.1.3680043.8.498.99872725822635667590591318423034454433","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11604298807403466096392316117803661479","series":"1.2.826.0.1.3680043.8.498.59353327480788491364091219525597036743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552","series":"1.2.826.0.1.3680043.8.498.54546083378381984787407203285582132897","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849","series":"1.2.826.0.1.3680043.8.498.12103311282035389519321944384948209864","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397","series":"1.2.826.0.1.3680043.8.498.11006727654624315522471736547261328631","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11642228346306175407288158335830376197","series":"1.2.826.0.1.3680043.8.498.10877992639099702182770139967917740923","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181","series":"1.2.826.0.1.3680043.8.498.10596270434001697223042523325218527920","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843","series":"1.2.826.0.1.3680043.8.498.87000889100561403251743606705928489251","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093","series":"1.2.826.0.1.3680043.8.498.20225926108662453431931503234943307046","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11669120316759278775792190599036288128","series":"1.2.826.0.1.3680043.8.498.11466574209831122744935397539920827927","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.32682907972361553476494033976263816809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11685904538570360040472064238371379681","series":"1.2.826.0.1.3680043.8.498.34008085572521269239087047000994035521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11691245281415120685024733512803575288","series":"1.2.826.0.1.3680043.8.498.66105449433257245605538649562709622572","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11703195077261884009442719578918319962","series":"1.2.826.0.1.3680043.8.498.29855462953845010031276834438087976321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11716605112302071535035323666308656381","series":"1.2.826.0.1.3680043.8.498.12271397768324503279137386118701839224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11720535027022778971695130536590302635","series":"1.2.826.0.1.3680043.8.498.12907340679366653683030355388707016805","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612","series":"1.2.826.0.1.3680043.8.498.21473688569387446452177927160000453269","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11725843001854130477287484865097535511","series":"1.2.826.0.1.3680043.8.498.63583288471734972919908861145629352914","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607","series":"1.2.826.0.1.3680043.8.498.98555813812900377524347193614735990949","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695","series":"1.2.826.0.1.3680043.8.498.52924740404777047342270508862630082827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950","series":"1.2.826.0.1.3680043.8.498.82363756536620643172728963235346808028","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11776647434143676427722616249587614803","series":"1.2.826.0.1.3680043.8.498.14365553184883150664804769997041665532","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11783597160745641720078978406968189488","series":"1.2.826.0.1.3680043.8.498.14728367232451279564394576090711163026","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269","series":"1.2.826.0.1.3680043.8.498.18419440817102039186858512078689980731","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11791567830371683660578052737702299124","series":"1.2.826.0.1.3680043.8.498.42149203628164150568880839675609919499","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762","series":"1.2.826.0.1.3680043.8.498.61090489198577885036259528844075288152","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11803762390999881908101130422708589537","series":"1.2.826.0.1.3680043.8.498.48392829180599992161046683989415282996","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991","series":"1.2.826.0.1.3680043.8.498.39611129203054820242555374068059896185","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969","series":"1.2.826.0.1.3680043.8.498.97839781672041209962373902597617709176","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200","series":"1.2.826.0.1.3680043.8.498.11083351605337075770250709857628080234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417","series":"1.2.826.0.1.3680043.8.498.53414807730948829036778780501957220845","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11860731961337988611847638299761007498","series":"1.2.826.0.1.3680043.8.498.28204227330866790795211780705847522140","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697","series":"1.2.826.0.1.3680043.8.498.87791029509082831482977468455719167093","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.41988546893959391510407149868806896390","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11881735311125899170335963782017197184","series":"1.2.826.0.1.3680043.8.498.64866236535776191115859055205894427739","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11892908237449514945544380023820626319","series":"1.2.826.0.1.3680043.8.498.13181819947563849545717762307293204437","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673","series":"1.2.826.0.1.3680043.8.498.18225856908710909605728128704869844805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369","series":"1.2.826.0.1.3680043.8.498.11707586665282925558585453385941173449","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11909779134152823550896593167929611759","series":"1.2.826.0.1.3680043.8.498.84982877557446192128274481285804561487","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851","series":"1.2.826.0.1.3680043.8.498.71987421900177276165276995848777866185","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11919420571588879483419578078755803351","series":"1.2.826.0.1.3680043.8.498.82264408332345385197032726261899556939","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11920531053482540304184657218030242597","series":"1.2.826.0.1.3680043.8.498.36215673402950310587789228112501361784","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028","series":"1.2.826.0.1.3680043.8.498.12622941360369335352328184844963874146","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.65844910056588489175130375327785747918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140","series":"1.2.826.0.1.3680043.8.498.66233048386965600072193483400905793639","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769","series":"1.2.826.0.1.3680043.8.498.12545239919751739193129050703745175704","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277","series":"1.2.826.0.1.3680043.8.498.11343344622343312110759493227556597110","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317","series":"1.2.826.0.1.3680043.8.498.28409850781395999697603052915910251457","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11979034558288454336812613626594510186","series":"1.2.826.0.1.3680043.8.498.66854834900828072887707848024232279019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970","series":"1.2.826.0.1.3680043.8.498.62820645828524135779219892485665966595","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11991512352474459753052051071717622771","series":"1.2.826.0.1.3680043.8.498.10233161898816834853452595516225064728","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11999008448480965933785988718780864948","series":"1.2.826.0.1.3680043.8.498.88527805564018949234362622005652741652","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878","series":"1.2.826.0.1.3680043.8.498.36766376930240933666381230276913958998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084","series":"1.2.826.0.1.3680043.8.498.50823238587101586335237204227580429849","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12015406188790105426257083591807220760","series":"1.2.826.0.1.3680043.8.498.99491484753875155708841518917210962011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12034579945815402722591687609951730205","series":"1.2.826.0.1.3680043.8.498.85308800877057054122066647360743133996","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12041392088066705782960819598258792338","series":"1.2.826.0.1.3680043.8.498.56381724755973043230161074873752613779","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179","series":"1.2.826.0.1.3680043.8.498.86643373703080488194155952873077944827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12054099347723802635102026371218576198","series":"1.2.826.0.1.3680043.8.498.83608977688607690993706580540041360788","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12067625141402105065337444349201280606","series":"1.2.826.0.1.3680043.8.498.46810323340285571161800733440702240586","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12071998470084955703027522152040042364","series":"1.2.826.0.1.3680043.8.498.41528672318554861840922386799986857724","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491","series":"1.2.826.0.1.3680043.8.498.72992997928004163031465132625232119439","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515","series":"1.2.826.0.1.3680043.8.498.12415702041327320940524222942427871473","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051","series":"1.2.826.0.1.3680043.8.498.69565590301244636241519819672683562641","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12130605331118953242364684736005454581","series":"1.2.826.0.1.3680043.8.498.59535920928440388209878697425271097435","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12131866860554871145836544571379633595","series":"1.2.826.0.1.3680043.8.498.96195328937473777043009281743778643703","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614","series":"1.2.826.0.1.3680043.8.498.55363145692412478780581973043871525354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.13348101806553877324577421903421844186","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12154188739734505793699974626991872621","series":"1.2.826.0.1.3680043.8.498.37165823890385469443493640446924800526","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169","series":"1.2.826.0.1.3680043.8.498.63024534917580398605655448356555992874","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12166235678892893545054546864889743251","series":"1.2.826.0.1.3680043.8.498.11157404538844182812228964380050124063","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851","series":"1.2.826.0.1.3680043.8.498.43818070203813490028408561368715633576","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581","series":"1.2.826.0.1.3680043.8.498.21046616680356460163440654447874624165","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211","series":"1.2.826.0.1.3680043.8.498.81192357764559453003810773433705406850","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12185615973620890770158935333033805789","series":"1.2.826.0.1.3680043.8.498.99463040379296740704232119345148212083","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408","series":"1.2.826.0.1.3680043.8.498.13073701439499115988339765864545111853","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360","series":"1.2.826.0.1.3680043.8.498.25858297814643222064173024168916981497","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796","series":"1.2.826.0.1.3680043.8.498.13146690352677049123645196993014010887","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968","series":"1.2.826.0.1.3680043.8.498.69385328132239523165254600258332411939","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12219658385745696203053079500246399481","series":"1.2.826.0.1.3680043.8.498.96794451467110213405036027057321768155","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.29727023765439474464302324612664820608","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.10756811194153083532992026997032203235","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.92537877606998871109135970378645056107","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817","series":"1.2.826.0.1.3680043.8.498.12361403760858458165556495989568081707","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392","series":"1.2.826.0.1.3680043.8.498.17832775036500137986592761825308530572","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12257119852150088212071954292747252243","series":"1.2.826.0.1.3680043.8.498.60832845930611806171842004814726666181","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12260661730837055872432103268905182876","series":"1.2.826.0.1.3680043.8.498.99630072206162873484224403731101411095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12264607993101717201407955391437731134","series":"1.2.826.0.1.3680043.8.498.85739239390779948779614184987386688628","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12266644932420958618394322959047940542","series":"1.2.826.0.1.3680043.8.498.10662818358197251991548987436519263581","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12271616365050313084031931009756086606","series":"1.2.826.0.1.3680043.8.498.10203993731461281855422323151721341873","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249","series":"1.2.826.0.1.3680043.8.498.11116380381696290455585051338120673645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.61708957883181496207022279314824901625","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.12587759538801636343464859863664401708","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12286894057773964720268768529280193085","series":"1.2.826.0.1.3680043.8.498.13380335532128404452665739508984932272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295443776523723913480216844110882937","series":"1.2.826.0.1.3680043.8.498.69163612007074033350916745634100177630","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12300749089664733313051543117302244508","series":"1.2.826.0.1.3680043.8.498.43617742555961091416251166661829096472","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199","series":"1.2.826.0.1.3680043.8.498.89738555185818812928997374017451342360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632","series":"1.2.826.0.1.3680043.8.498.99960433273557168945332980270758348163","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097","series":"1.2.826.0.1.3680043.8.498.10914156777786339778162233112544582288","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12328313245396930020974053819810033945","series":"1.2.826.0.1.3680043.8.498.68217436167769410424406041962945255351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081","series":"1.2.826.0.1.3680043.8.498.33994302602647933128707045029806574972","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12345064031074704673835101507426902246","series":"1.2.826.0.1.3680043.8.498.82232643011613587776822463517223473663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348875283993064999589090039392140219","series":"1.2.826.0.1.3680043.8.498.71249957090963399057243337487375579401","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12349656752404066719917550972333542508","series":"1.2.826.0.1.3680043.8.498.95193839623005132269966440289165875440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12355011301785011370279005103251815187","series":"1.2.826.0.1.3680043.8.498.28184593840512391843202949140810117710","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269","series":"1.2.826.0.1.3680043.8.498.57064911818345480379728841760760023979","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380","series":"1.2.826.0.1.3680043.8.498.20463391854947025180034027215961084865","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636","series":"1.2.826.0.1.3680043.8.498.30781469089524084637123252696935909969","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12381909868368799280462187344958674397","series":"1.2.826.0.1.3680043.8.498.98772572652731378653838460767644931757","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387075993162655781372091860026675548","series":"1.2.826.0.1.3680043.8.498.81354764445645298985479892891572350620","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387667155732116394489547491673280062","series":"1.2.826.0.1.3680043.8.498.97420399694994614891008872887151598216","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894","series":"1.2.826.0.1.3680043.8.498.75158129661235577023930763597325498729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408314059030096510457440879771212020","series":"1.2.826.0.1.3680043.8.498.65298383245218592734618206813776776580","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408945963789246646229147558387793270","series":"1.2.826.0.1.3680043.8.498.34975249118332179190340489759967248596","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12414869188691502885668778427120766321","series":"1.2.826.0.1.3680043.8.498.76224102521865456083438413977533220982","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12432974885645948103184031984290883090","series":"1.2.826.0.1.3680043.8.498.65486216070199521417545028267318591379","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12437154858167700438432674983971817217","series":"1.2.826.0.1.3680043.8.498.13190130174417231523654547881587195562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945","series":"1.2.826.0.1.3680043.8.498.80909030645776100335455943053491814088","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656","series":"1.2.826.0.1.3680043.8.498.44201953521249266558147170632355365292","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896","series":"1.2.826.0.1.3680043.8.498.97413504911390341527834676168125961337","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12481913908649439189738319460342358360","series":"1.2.826.0.1.3680043.8.498.92778641032167107247147544318615321923","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12487362549907644519947672428024977201","series":"1.2.826.0.1.3680043.8.498.11037148382268585141931484327030464882","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12490034912643658601552092574630438642","series":"1.2.826.0.1.3680043.8.498.37298831497715193767659357462000856975","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491747080684524905643593142141672951","series":"1.2.826.0.1.3680043.8.498.10962266160953754876775344112062053828","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525","series":"1.2.826.0.1.3680043.8.498.63575267426050513291666795133819455482","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038","series":"1.2.826.0.1.3680043.8.498.20059995939072300652312755832382885577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696","series":"1.2.826.0.1.3680043.8.498.15600074407225031754302491905788308981","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12509944331769453135724043962410129357","series":"1.2.826.0.1.3680043.8.498.10508656100735574847807155638319534645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12515322634403509077334846234426636136","series":"1.2.826.0.1.3680043.8.498.11256127730205537607690552032136076344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342","series":"1.2.826.0.1.3680043.8.498.54241873639640878759638315853140966046","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12526590714362529606046044104426807158","series":"1.2.826.0.1.3680043.8.498.13252007445288356405323242831577384058","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12531442175319757740745664049001806494","series":"1.2.826.0.1.3680043.8.498.10029797422478509094784245341164202835","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12534628800787651008876070514080610643","series":"1.2.826.0.1.3680043.8.498.13270106397385244325344594197927892666","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12548176380550230498070609559486512004","series":"1.2.826.0.1.3680043.8.498.10645385620204048290942200086497488562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12553455350653672209409408595948190982","series":"1.2.826.0.1.3680043.8.498.11970003939626181119744013678530304379","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12559687473583953266143897736224215680","series":"1.2.826.0.1.3680043.8.498.80403581402260561238459876130882112678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12569762036474456642024618586300354737","series":"1.2.826.0.1.3680043.8.498.53273654929983228974566670201659870185","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785","series":"1.2.826.0.1.3680043.8.498.12950449425329234901569600880880957465","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298","series":"1.2.826.0.1.3680043.8.498.12889804120947853135974851021000509714","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184","series":"1.2.826.0.1.3680043.8.498.66867725389593247784367071445863334495","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691","series":"1.2.826.0.1.3680043.8.498.12582497901289250157974715882272899395","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12607203615180268117341271176820357242","series":"1.2.826.0.1.3680043.8.498.15563016142287247352911382199307635606","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12614097867016983076979065551985251584","series":"1.2.826.0.1.3680043.8.498.63808550926073464665838746113871294015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842","series":"1.2.826.0.1.3680043.8.498.18755445171771574642694282278497530105","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12626383124142146001532197239671686442","series":"1.2.826.0.1.3680043.8.498.16711279084694447046322147305274546090","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12630423349626505319792444053874011718","series":"1.2.826.0.1.3680043.8.498.84763003080304144344972352148816574143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491","series":"1.2.826.0.1.3680043.8.498.41655394541483878359835350726778538397","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582","series":"1.2.826.0.1.3680043.8.498.52865871330972672365946425900496715852","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635","series":"1.2.826.0.1.3680043.8.498.11045576656606769904599839922474016717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12655324218817524683138594937984186560","series":"1.2.826.0.1.3680043.8.498.40994213282963723688744432906349130304","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12660901572462562819575002681831361427","series":"1.2.826.0.1.3680043.8.498.79958422089386099850296479042738504137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095","series":"1.2.826.0.1.3680043.8.498.51881879771586296768304161516669165175","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12676359576477452816896379524598090410","series":"1.2.826.0.1.3680043.8.498.32070626227873150622142355930905577089","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12690080124553846006638565600585469969","series":"1.2.826.0.1.3680043.8.498.30359662891644580678585466843823459947","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922","series":"1.2.826.0.1.3680043.8.498.29998840657737203171910520380027546904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12700591955254168977239658917260628326","series":"1.2.826.0.1.3680043.8.498.25555906204383124303071210447291213863","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12704063145595847725421427571860589471","series":"1.2.826.0.1.3680043.8.498.19024778338475549440074923953888152897","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715130507546263968763238702574191950","series":"1.2.826.0.1.3680043.8.498.11246259618708651009476691061122545567","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849","series":"1.2.826.0.1.3680043.8.498.99742552647138207772944162520167143655","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688","series":"1.2.826.0.1.3680043.8.498.32193924410879680452748154173544096594","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122","series":"1.2.826.0.1.3680043.8.498.98909230041787426426300942109471906057","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12728397615513347615111391662798012012","series":"1.2.826.0.1.3680043.8.498.38765716066934700347036610415453013010","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067","series":"1.2.826.0.1.3680043.8.498.96154335942332055089378317225227749496","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12762076104929080571464647825161185643","series":"1.2.826.0.1.3680043.8.498.78014944424273760301383876790213156345","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12765434812409174436720333864278447770","series":"1.2.826.0.1.3680043.8.498.11901673435198341845898956964320977795","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535","series":"1.2.826.0.1.3680043.8.498.32595801313895338807784020561006035119","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12775105605170844918620305197829216665","series":"1.2.826.0.1.3680043.8.498.11649135246815831359278832664355731174","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180","series":"1.2.826.0.1.3680043.8.498.12653531952401174431661536918007095083","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785792890626338008419452097492192088","series":"1.2.826.0.1.3680043.8.498.79417746325238434806767453608883753312","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212","series":"1.2.826.0.1.3680043.8.498.24839099855144317047550172273239799357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358","series":"1.2.826.0.1.3680043.8.498.12340061439866170118186800590627959453","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12806792356922865929896103306611295151","series":"1.2.826.0.1.3680043.8.498.14218970577913620022526353072485088852","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652","series":"1.2.826.0.1.3680043.8.498.46386595347270493691079799361342548074","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046","series":"1.2.826.0.1.3680043.8.498.55521726295294212747730960537225630834","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569","series":"1.2.826.0.1.3680043.8.498.71025802642727941731802898978385344707","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12827120140904847104302376671076396695","series":"1.2.826.0.1.3680043.8.498.12617652317832066644748148831100714213","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12836702174475748025129787471026627810","series":"1.2.826.0.1.3680043.8.498.20740918137165374791865004347146891949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12862782592303338109631225504988756108","series":"1.2.826.0.1.3680043.8.498.80497338217356011526281188664963830642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655","series":"1.2.826.0.1.3680043.8.498.11730565269661595214493592709095505618","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608","series":"1.2.826.0.1.3680043.8.498.64919349678698050772403188166009561326","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.88999280746836672639951329293565892460","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.42728894453720330458191403252985663444","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.65954429293178706019862895708841923460","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12938500386086171684983091332136233036","series":"1.2.826.0.1.3680043.8.498.13189679200479811071549834860485771476","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544","series":"1.2.826.0.1.3680043.8.498.22610612596740933854562901054309655952","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12950639892345812884896649629409199101","series":"1.2.826.0.1.3680043.8.498.80384476626163060628331950179122103552","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111","series":"1.2.826.0.1.3680043.8.498.49321689511542424358257904173242980943","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12961116776122838195555132309258963046","series":"1.2.826.0.1.3680043.8.498.12006172447578015165768631603173370859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12970315889871740202726744767179863179","series":"1.2.826.0.1.3680043.8.498.61439685417271915872562631717507972886","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.70685133098265812758625746226740328804","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172","series":"1.2.826.0.1.3680043.8.498.52116520815829513058701049694276122225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12986028147682493711912242081052787346","series":"1.2.826.0.1.3680043.8.498.78796094223525918081758501420836735128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12992013816638129896738209015383985569","series":"1.2.826.0.1.3680043.8.498.48981345513954196088985077674621731339","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808","series":"1.2.826.0.1.3680043.8.498.64172525187784476831767988039843788894","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13015502057686077045104167339480215528","series":"1.2.826.0.1.3680043.8.498.43084125425105237760465000827878912725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13022266764623819477228425331705640702","series":"1.2.826.0.1.3680043.8.498.42426012066454491669490542626071485233","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13024625496981667037628275235696732200","series":"1.2.826.0.1.3680043.8.498.54654971805106970731631945721180467142","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594","series":"1.2.826.0.1.3680043.8.498.55074641293921305422186221798611534478","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589","series":"1.2.826.0.1.3680043.8.498.12378521210323504769562324379693048299","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229","series":"1.2.826.0.1.3680043.8.498.71179133107147914995816648166648505980","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999","series":"1.2.826.0.1.3680043.8.498.78472201377946920330173668946782220679","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508","series":"1.2.826.0.1.3680043.8.498.10892364241387013516930423928687378158","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974","series":"1.2.826.0.1.3680043.8.498.30226348050479539331924289768648692265","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13073387272228394257811861905056479438","series":"1.2.826.0.1.3680043.8.498.82329733572189389756224735983332914535","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703","series":"1.2.826.0.1.3680043.8.498.10425462399662448096411573966479337214","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13079170182386765602129976624023222742","series":"1.2.826.0.1.3680043.8.498.87232667841602520374947593744241399932","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13087770868099843948960419968576882974","series":"1.2.826.0.1.3680043.8.498.10895457057366871216305851930259497030","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13092094949394905924922534008771583745","series":"1.2.826.0.1.3680043.8.498.62128077643613414118895966733104799919","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987","series":"1.2.826.0.1.3680043.8.498.98134175580050576296407459711827819078","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786","series":"1.2.826.0.1.3680043.8.498.12433274967393258513446013534525165916","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13113623966675790946816351311283701141","series":"1.2.826.0.1.3680043.8.498.23005716277623560335556340362528194058","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127","series":"1.2.826.0.1.3680043.8.498.11257061824962402087005848021800501752","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13118532243906060645114391749986421707","series":"1.2.826.0.1.3680043.8.498.92502737523192082435615200592591097475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.19610019955974063557448246438051187023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142","series":"1.2.826.0.1.3680043.8.498.34893343144142950600433300301192221612","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13130534736322715018024786213131944607","series":"1.2.826.0.1.3680043.8.498.75531033581531858148001831351592093209","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660","series":"1.2.826.0.1.3680043.8.498.10568066561746571915886159307182011459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13136367866260644484250529322937554927","series":"1.2.826.0.1.3680043.8.498.24154903485090332509949937608704262504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13137549927896953291645459480590055925","series":"1.2.826.0.1.3680043.8.498.10968873836794516554067290658492501243","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116","series":"1.2.826.0.1.3680043.8.498.12597809182422921706503466922417803251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13148805278538458401210722493048566510","series":"1.2.826.0.1.3680043.8.498.94005116493715554005973441529135805170","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13150794288945922409613136250303663502","series":"1.2.826.0.1.3680043.8.498.12230536477872701916852281189275395581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13155572156594271049520969168607160312","series":"1.2.826.0.1.3680043.8.498.12740973054582975723630950986782120967","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.10622847405565705403945786287990550954","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13162284470388627411596358366248494046","series":"1.2.826.0.1.3680043.8.498.58493843372764224740666920143593944649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13165770394160595162984722917452661205","series":"1.2.826.0.1.3680043.8.498.13357425259893315140064462993396148890","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13172295193446466338347273836250944090","series":"1.2.826.0.1.3680043.8.498.14440416242434302738906958626651365046","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710","series":"1.2.826.0.1.3680043.8.498.11687778657900445367965481071149868019","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13179169600887013946657470830682778717","series":"1.2.826.0.1.3680043.8.498.62911169175387949149488794950056235587","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13199635020145400856987271814458645896","series":"1.2.826.0.1.3680043.8.498.26268885107186369796927948794125077670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495","series":"1.2.826.0.1.3680043.8.498.75995422442957411485267699088650075294","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.10408989185029972401995366241754102474","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13214267416673058581197714011913363750","series":"1.2.826.0.1.3680043.8.498.90481900496778587123354074405427736038","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226","series":"1.2.826.0.1.3680043.8.498.61567047363331623669592893816171454234","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13232501202175905676335982676386953813","series":"1.2.826.0.1.3680043.8.498.11487417829567667163229049931264315026","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13240749152416621711688053465035013133","series":"1.2.826.0.1.3680043.8.498.44500152077279558352537801097941270374","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501","series":"1.2.826.0.1.3680043.8.498.61916284251743357884889923431099768277","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025","series":"1.2.826.0.1.3680043.8.498.84718863918876400686050945247678782282","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262583253812134425581922368942846684","series":"1.2.826.0.1.3680043.8.498.21593656634491434991163420175753315672","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891","series":"1.2.826.0.1.3680043.8.498.73978745362309858790387708743520311872","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13273890409523605456488117581706115075","series":"1.2.826.0.1.3680043.8.498.48302873127639605262688812585368358409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634","series":"1.2.826.0.1.3680043.8.498.15349536531881217555964707474035504950","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13288244567806030859347115222642155186","series":"1.2.826.0.1.3680043.8.498.50661677208857696998003933729132647273","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156","series":"1.2.826.0.1.3680043.8.498.40324125648836921987748628020869650813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518","series":"1.2.826.0.1.3680043.8.498.97420238106950882785296111648154930946","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.20101400503182664594192174876979379111","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13321746459276191365724686301267421531","series":"1.2.826.0.1.3680043.8.498.10056920235938228721112806853631031057","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13328029743559327467301971917273397645","series":"1.2.826.0.1.3680043.8.498.30300733304618293199468674541144393263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903","series":"1.2.826.0.1.3680043.8.498.68216362009231374856176479984344037705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030","series":"1.2.826.0.1.3680043.8.498.52571108109766329747936233936395753931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353966209257569249620562863006006441","series":"1.2.826.0.1.3680043.8.498.78734311110769661097016986391977466805","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604","series":"1.2.826.0.1.3680043.8.498.49906054911215250054815883889478324340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13363856251607349805111249441972460575","series":"1.2.826.0.1.3680043.8.498.88486939780950516884676034520442725620","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388","series":"1.2.826.0.1.3680043.8.498.87312074820099057827775180456044065431","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13371001997511542011416510364359588752","series":"1.2.826.0.1.3680043.8.498.22021570052701125698129186741628936625","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141","series":"1.2.826.0.1.3680043.8.498.56807194747701013272480306696551349931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13391735462825209934041441767886593395","series":"1.2.826.0.1.3680043.8.498.10217649962990978089530568109904968250","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13399254446600380420635633478666850540","series":"1.2.826.0.1.3680043.8.498.43954172762427362769898771921347945985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.11121266312276659683468880037211880848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13455212888114594010716499984611216646","series":"1.2.826.0.1.3680043.8.498.29576281506294092651716782439766742893","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.82569189050563393890210481562695236315","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185","series":"1.2.826.0.1.3680043.8.498.72854672927998858684227481092484697692","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226","series":"1.2.826.0.1.3680043.8.498.10072118276867498117556502528770891174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13660439785395658826946769630921591235","series":"1.2.826.0.1.3680043.8.498.93395083596331362657089247880819902217","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13737588520456064856233576393747389054","series":"1.2.826.0.1.3680043.8.498.23134223305027068697688765007706351666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13843368750899904654034226115826130084","series":"1.2.826.0.1.3680043.8.498.11025636651209364786193881783451070516","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13915367834258386094001181500560177470","series":"1.2.826.0.1.3680043.8.498.32118559313451399836691814908124865513","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13975027824128345697756313869165838055","series":"1.2.826.0.1.3680043.8.498.45133718767024782194516783521288927096","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791","series":"1.2.826.0.1.3680043.8.498.68222111798068082084655654690078121587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596","series":"1.2.826.0.1.3680043.8.498.48023464407504409225343537792943296339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704","series":"1.2.826.0.1.3680043.8.498.10453101273293799976518158592893169210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14236519693915284584208989450687591759","series":"1.2.826.0.1.3680043.8.498.68309257162901271908542022072121353067","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838","series":"1.2.826.0.1.3680043.8.498.11713481291666199097582691235100273383","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14377917653321813299237274871837695586","series":"1.2.826.0.1.3680043.8.498.82413676475276021802651997866155203641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.63296658633482703173315697924995518270","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268","series":"1.2.826.0.1.3680043.8.498.12901226097432700042740819606965614119","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14511862461248267537051973802039959211","series":"1.2.826.0.1.3680043.8.498.18677245736122393094703037422951154254","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14679140388328648617386311272361396555","series":"1.2.826.0.1.3680043.8.498.51834226559068471500245103788911169978","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570","series":"1.2.826.0.1.3680043.8.498.57781237231925934815208630072175328012","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14806659119591146961784818305031900863","series":"1.2.826.0.1.3680043.8.498.36576398008482202836742702151087685754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14958560978176121375035839707004667832","series":"1.2.826.0.1.3680043.8.498.12101052819472792275620400020476214250","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069","series":"1.2.826.0.1.3680043.8.498.95561123268974527381260528092912399986","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.13227156330997586585940293461814122733","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15090991392324602559982474708414427621","series":"1.2.826.0.1.3680043.8.498.11714609993464474965966079546398363141","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15219410096632334470101257478715035071","series":"1.2.826.0.1.3680043.8.498.11652687734830090584384538883393213131","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15297444019822621504849851092382560980","series":"1.2.826.0.1.3680043.8.498.81796630571430601799074946979320473209","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","series":"1.2.826.0.1.3680043.8.498.85209447296560548781965949457490592337","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885","series":"1.2.826.0.1.3680043.8.498.79812752664946333316235005213307604576","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15444316810805746453392291780593934328","series":"1.2.826.0.1.3680043.8.498.36789968901856227949370186210551992272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397","series":"1.2.826.0.1.3680043.8.498.90729909943541777278399310343577974575","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309","series":"1.2.826.0.1.3680043.8.498.55399266565350183802077573346939446216","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742","series":"1.2.826.0.1.3680043.8.498.11441797997693848893576397848979022654","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.11234742990571523936612291006495444865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764","series":"1.2.826.0.1.3680043.8.498.71335309655399571848664085478019540297","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16051798246004436152597344963060310360","series":"1.2.826.0.1.3680043.8.498.10071731081841776926911954285395913583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094","series":"1.2.826.0.1.3680043.8.498.64697356717312527393176728436388958424","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832","series":"1.2.826.0.1.3680043.8.498.14490902795096723892308052655753771015","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16258190393569006141285550986533848849","series":"1.2.826.0.1.3680043.8.498.80057585976582795150467431201857183260","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16338296537380784459604897543069625390","series":"1.2.826.0.1.3680043.8.498.12416262264255390094120364143782609193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331","series":"1.2.826.0.1.3680043.8.498.84969344232558856726022533707391382236","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16483470188406187000599644150600279596","series":"1.2.826.0.1.3680043.8.498.87850215334305755323661464864842913605","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.20815875871582615712806510216364429280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16581426215518472805200227992550139190","series":"1.2.826.0.1.3680043.8.498.40201593450811824869489904903039233504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849","series":"1.2.826.0.1.3680043.8.498.28476398180322790325431707288703199421","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16691383858003633107641297162554868325","series":"1.2.826.0.1.3680043.8.498.67335441164844698451715816282952347740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16718788321951483456195507236392554704","series":"1.2.826.0.1.3680043.8.498.23136975604751406777052922144053388149","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","series":"1.2.826.0.1.3680043.8.498.77207325005228809695380601793615826402","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16845451648760595922758166878121325861","series":"1.2.826.0.1.3680043.8.498.46297175789577846950046172944705142424","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16886658450293145802897654768506439494","series":"1.2.826.0.1.3680043.8.498.12155482768149201736168190320008865211","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817","series":"1.2.826.0.1.3680043.8.498.11310054808480957081341510016335965959","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.33643561168461023721936091827199577978","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17107696939614797679012793547913769707","series":"1.2.826.0.1.3680043.8.498.89092936310103841170691209866829204410","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17146868851921828837871671912815843012","series":"1.2.826.0.1.3680043.8.498.85297535217843228765328146948973015558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17265093113586420061761042467476063876","series":"1.2.826.0.1.3680043.8.498.71398888383696526359498669870567261151","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17336223563862025807066700226971845206","series":"1.2.826.0.1.3680043.8.498.75060258972335767133194178295119754140","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17415936504684075438800895036740822145","series":"1.2.826.0.1.3680043.8.498.63995757305452373523445052239997720872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17517950899674150719043097786728121608","series":"1.2.826.0.1.3680043.8.498.10546926949877168204494874973711987352","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319","series":"1.2.826.0.1.3680043.8.498.94137563787345184996737983116755537359","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17663005209256897892831319144686413853","series":"1.2.826.0.1.3680043.8.498.12162127607634972851898381243145573191","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.13373776628240438275030497598409258470","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17813878802252660525798053577039470087","series":"1.2.826.0.1.3680043.8.498.11312375484740331444545859566243567873","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916","series":"1.2.826.0.1.3680043.8.498.10648055698380555450770989280200929187","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17909590711893200514107254988453660857","series":"1.2.826.0.1.3680043.8.498.33746053796891714209466244408646741045","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467","series":"1.2.826.0.1.3680043.8.498.77154390264686577448531898169503992373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18090147030578547920423840630147682826","series":"1.2.826.0.1.3680043.8.498.12943020891195253245429903353410759499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18113123512696698441392990570166201548","series":"1.2.826.0.1.3680043.8.498.48133754883459093795140678383758807762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18176586296602067352845573876750324761","series":"1.2.826.0.1.3680043.8.498.95135380656177251310533538187272813610","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18230847745299902998425838090653825745","series":"1.2.826.0.1.3680043.8.498.60733372717579061395537203941467678670","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569","series":"1.2.826.0.1.3680043.8.498.81052757011658416365412087413360887136","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378","series":"1.2.826.0.1.3680043.8.498.30573310138109979980328563769756776360","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18509793799833391995987656714798904411","series":"1.2.826.0.1.3680043.8.498.31402757045664401022355136940296765727","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18592535981451766048952107991674227570","series":"1.2.826.0.1.3680043.8.498.92717273729883882352176781573670214859","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18625377919309205857120774497713011583","series":"1.2.826.0.1.3680043.8.498.11570813469993702740198301157345123499","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829","series":"1.2.826.0.1.3680043.8.498.57525924902205150544248843434006557401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18721466442792788108431096912844719622","series":"1.2.826.0.1.3680043.8.498.75378336926241467514388366463040267329","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18792568294433896490648198192877693850","series":"1.2.826.0.1.3680043.8.498.59277752391225047203460273436877588732","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18825393854436144614202928260751418496","series":"1.2.826.0.1.3680043.8.498.92590983861801896230209602015487303902","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18899625751876258520147203904985857313","series":"1.2.826.0.1.3680043.8.498.64158267422612640026701150826710275773","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874","series":"1.2.826.0.1.3680043.8.498.65299369755140102624232399807882184290","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19104350033985745058396432133828748480","series":"1.2.826.0.1.3680043.8.498.56355839812504047455304484894692082648","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19178214677492694135853596920291058397","series":"1.2.826.0.1.3680043.8.498.14696050235366351219018553994428929032","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19197037365998103318720742154440921358","series":"1.2.826.0.1.3680043.8.498.12535713663960308615370644753291474374","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19291375342005405073321673638971145777","series":"1.2.826.0.1.3680043.8.498.13074572341579792326032583749320722989","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841","series":"1.2.826.0.1.3680043.8.498.12554153563601248448420871595479748354","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19486520578867272329833751210055639515","series":"1.2.826.0.1.3680043.8.498.55556918162081522878381246711154691288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182","series":"1.2.826.0.1.3680043.8.498.10167955056646179667609464919974025593","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19592989024994864431870593504844309885","series":"1.2.826.0.1.3680043.8.498.10999243104292802162073528803301708691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310","series":"1.2.826.0.1.3680043.8.498.65875595848746561757418562875055234549","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19788612491138642031531719942303619648","series":"1.2.826.0.1.3680043.8.498.62337200027316955091925223027834174230","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246","series":"1.2.826.0.1.3680043.8.498.12573624812133653523205747144146765752","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530","series":"1.2.826.0.1.3680043.8.498.23385529225779993097024888805820360059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19895710326297085463251295629948896948","series":"1.2.826.0.1.3680043.8.498.58239794296938648330610814512490229054","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19965044457637713843792494926652048462","series":"1.2.826.0.1.3680043.8.498.60025353697014161375899969150850062015","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390","series":"1.2.826.0.1.3680043.8.498.97649334101626366289724790464760926261","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830","series":"1.2.826.0.1.3680043.8.498.74746365934563371258223236405420153893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20192024727432864451309151144591371229","series":"1.2.826.0.1.3680043.8.498.57847406871065276072312100018568320672","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284","series":"1.2.826.0.1.3680043.8.498.85580698730087564978522604357205465891","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20269296702024202621734017056347194499","series":"1.2.826.0.1.3680043.8.498.94284416663605591206297395908976513563","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20411068889491114730824265411248922690","series":"1.2.826.0.1.3680043.8.498.13166244367029247756201987869407112427","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372","series":"1.2.826.0.1.3680043.8.498.11925328941814705548649877113404219773","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886","series":"1.2.826.0.1.3680043.8.498.71776347363552533436056527036005244084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20522330861615570894210641293344468905","series":"1.2.826.0.1.3680043.8.498.46782749961303965962548940093204599690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785","series":"1.2.826.0.1.3680043.8.498.27656020313463557720472797504760713810","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20733151481921261415295204209910573639","series":"1.2.826.0.1.3680043.8.498.26118892537419085557301539679548395641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20850785960533583051456342090773230767","series":"1.2.826.0.1.3680043.8.498.88721794171893739924932497584865191321","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537","series":"1.2.826.0.1.3680043.8.498.73593863203374157535717281135637617510","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430","series":"1.2.826.0.1.3680043.8.498.10621633968174075869953411885289061506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954","series":"1.2.826.0.1.3680043.8.498.11252550908122112963048082445356671932","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21116046804798247223618072736212944615","series":"1.2.826.0.1.3680043.8.498.52761238265664543425030312857659066627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21239630702461934662855021164119114203","series":"1.2.826.0.1.3680043.8.498.44600399777031243230652596375848704698","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454","series":"1.2.826.0.1.3680043.8.498.12169592222988048509907969985357359146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283","series":"1.2.826.0.1.3680043.8.498.12309473356029514104187086447050128449","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958","series":"1.2.826.0.1.3680043.8.498.10190760646779643122650801767315825410","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21520719119040558609068952147312604007","series":"1.2.826.0.1.3680043.8.498.65402917600418835894906917079276481469","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030","series":"1.2.826.0.1.3680043.8.498.37602122132898681980613457197155766280","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21606196343708804451390109412118036330","series":"1.2.826.0.1.3680043.8.498.69758444945480502033227478842724221746","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137","series":"1.2.826.0.1.3680043.8.498.52340590126764900515760488243554108565","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437","series":"1.2.826.0.1.3680043.8.498.82825408311914096645205441803211679168","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673","series":"1.2.826.0.1.3680043.8.498.11349875172238633124990852670323266349","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.11503697292988964850537522482109804415","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295","series":"1.2.826.0.1.3680043.8.498.66629840789204122914903972284035684640","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22027914323057444746912555313058666950","series":"1.2.826.0.1.3680043.8.498.12162458340165737040461977183179644039","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22088734384869383587217556574382319654","series":"1.2.826.0.1.3680043.8.498.72477643008178998546651918738009102496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.25666140776715226080788997996627126775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22135282054716841452811687331325271982","series":"1.2.826.0.1.3680043.8.498.23094642084293848573222438499281135361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129","series":"1.2.826.0.1.3680043.8.498.13243160805922545284769276591868445553","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22357754074308826479280023393343731000","series":"1.2.826.0.1.3680043.8.498.27420792467883331262543574108784522968","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005","series":"1.2.826.0.1.3680043.8.498.71081959542479039008766877695344049217","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22426390571898968732573184721547490379","series":"1.2.826.0.1.3680043.8.498.16960014699163769841341040534701919683","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22480218016816209183384996714060953859","series":"1.2.826.0.1.3680043.8.498.38385958290037302131631259071754721121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128","series":"1.2.826.0.1.3680043.8.498.36300665521074994078164442160997227279","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22617494262793136174349268379191631799","series":"1.2.826.0.1.3680043.8.498.90226905938201588888119010377597329806","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22656222254273662937226093008714426490","series":"1.2.826.0.1.3680043.8.498.40479840276657580104096671725295847620","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","series":"1.2.826.0.1.3680043.8.498.70921189728814367182758099613470499122","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22886251629508830700774506697670139848","series":"1.2.826.0.1.3680043.8.498.11583796286461524679609141080591201528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170","series":"1.2.826.0.1.3680043.8.498.46377753465095808239640020762264758701","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134","series":"1.2.826.0.1.3680043.8.498.46659136368189211126461327449160298088","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23073694267101759922738121732233014980","series":"1.2.826.0.1.3680043.8.498.10885930455850527162892277026203191579","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23094733099955217022059584418468076048","series":"1.2.826.0.1.3680043.8.498.37063303453628436266144384141090896819","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23182005205627494484490011351673355568","series":"1.2.826.0.1.3680043.8.498.12639570143924236466836367398263128890","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932","series":"1.2.826.0.1.3680043.8.498.96043767365578405808497489886271788851","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288","series":"1.2.826.0.1.3680043.8.498.24763327400751927294064161675407464320","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663","series":"1.2.826.0.1.3680043.8.498.19881206539187265123945071601449576104","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23703817737293327679300330167234268929","series":"1.2.826.0.1.3680043.8.498.29733716378340779342861526015842498191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088","series":"1.2.826.0.1.3680043.8.498.10222138187662296716685243217835676138","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23833933822461809237193998300076847210","series":"1.2.826.0.1.3680043.8.498.11214114946358789493520068832453567580","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23911658810234305013984976739924353704","series":"1.2.826.0.1.3680043.8.498.73660640502280898255822238217879749471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23986051274386630509877205174270047762","series":"1.2.826.0.1.3680043.8.498.45486179012549663929458966088626205723","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530","series":"1.2.826.0.1.3680043.8.498.10973306044747059940542641637515070681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24209643225939781032698894992032106781","series":"1.2.826.0.1.3680043.8.498.13379198197879294076130413877134950088","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24267153760943721146187182369810590631","series":"1.2.826.0.1.3680043.8.498.22019065675595632769939308404873208724","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24291465113360632954006568021942978765","series":"1.2.826.0.1.3680043.8.498.18425223660531619972078807865702171417","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557","series":"1.2.826.0.1.3680043.8.498.67954101476322784463095191846767762397","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24478195415117310702720951368429189290","series":"1.2.826.0.1.3680043.8.498.67993818049945495892107988968613297580","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24555528844026121007453886375651403314","series":"1.2.826.0.1.3680043.8.498.82612917114757042901794207103941610890","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503","series":"1.2.826.0.1.3680043.8.498.83725936957424306595093730826762381061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24679905857546029995247839380830345841","series":"1.2.826.0.1.3680043.8.498.11083399466179773163137919808431813129","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038","series":"1.2.826.0.1.3680043.8.498.11142718040368118670814050326716297556","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684","series":"1.2.826.0.1.3680043.8.498.10749884339108343177912228690383368943","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187","series":"1.2.826.0.1.3680043.8.498.28078432782112988362615465750559327173","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25006652485159423287561656594171884310","series":"1.2.826.0.1.3680043.8.498.14511440724505552650919347127706140504","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674","series":"1.2.826.0.1.3680043.8.498.36610755000129381543973307033277748301","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047","series":"1.2.826.0.1.3680043.8.498.36126260752946615878331170504968497079","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25167060436407065195851060078175404789","series":"1.2.826.0.1.3680043.8.498.39311761807960026618721385472839681713","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25206096958520407302630747466352252249","series":"1.2.826.0.1.3680043.8.498.12373669675578100726022512449116424255","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25266231062774974303723359806535863810","series":"1.2.826.0.1.3680043.8.498.51334208902638504550921228530189719112","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25359770839263995354733835365510047109","series":"1.2.826.0.1.3680043.8.498.10891962874374730014804867587369713595","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25440342930186170531037102906687829898","series":"1.2.826.0.1.3680043.8.498.30315259523920384099247352871061391644","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25487225187305802150215261512092051199","series":"1.2.826.0.1.3680043.8.498.10663359560299628397809943179215083650","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882","series":"1.2.826.0.1.3680043.8.498.43891534129651583307168290100678295421","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25558526735614916746398919603493693652","series":"1.2.826.0.1.3680043.8.498.11429117708180669880403193859419794020","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505","series":"1.2.826.0.1.3680043.8.498.57637846215643520635274142834789344702","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25650384476012633612528314070894186451","series":"1.2.826.0.1.3680043.8.498.10685517006137178592841037805334433012","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.98900787858835708475221944393573606609","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629","series":"1.2.826.0.1.3680043.8.498.34526159429912223542060883648741564643","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096","series":"1.2.826.0.1.3680043.8.498.11499992333589408206255768377124855013","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260","series":"1.2.826.0.1.3680043.8.498.67161297521784910799333220675069677480","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25818541516137783435601858274495597736","series":"1.2.826.0.1.3680043.8.498.62853151767727481067428914076985434900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25904473139226718487560978103077541725","series":"1.2.826.0.1.3680043.8.498.33258788924077222303652041119370307607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036","series":"1.2.826.0.1.3680043.8.498.16580221580701245322193571809363922431","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26002972602557466222471260313257965551","series":"1.2.826.0.1.3680043.8.498.58103506318909758994944271084813649618","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604","series":"1.2.826.0.1.3680043.8.498.43862843238222020850000829832931919604","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26072622476167889001255150030772906081","series":"1.2.826.0.1.3680043.8.498.54208471513524592398018911681222609491","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26257425287121280175418060407044075705","series":"1.2.826.0.1.3680043.8.498.12983129180289081939194183880744431598","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26325951519908477683595799515094966069","series":"1.2.826.0.1.3680043.8.498.35455366870581331476227614183932879941","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730","series":"1.2.826.0.1.3680043.8.498.34569269094198269865160385324493649357","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.12926086015756624628977299572344280594","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26512333939158074470511885252122191522","series":"1.2.826.0.1.3680043.8.498.77460024413595212819735036393527845080","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26542435427465415008860869796504112727","series":"1.2.826.0.1.3680043.8.498.30291357178508942325311117222605811618","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112","series":"1.2.826.0.1.3680043.8.498.91159993076282188341661139199230292099","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26792665054823243997838322477587733719","series":"1.2.826.0.1.3680043.8.498.23522060408252475974603623597924394258","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26843088620128166809417361066548447201","series":"1.2.826.0.1.3680043.8.498.97657282609862798591221357045536726300","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601","series":"1.2.826.0.1.3680043.8.498.33242921367015950924783448667738519015","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004","series":"1.2.826.0.1.3680043.8.498.85375522012512874345306641580478105451","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27023892731021470476500135946317920771","series":"1.2.826.0.1.3680043.8.498.86527540727058879607191476275492866724","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194","series":"1.2.826.0.1.3680043.8.498.11709067061959921301709118093893621555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.11149206967333191886054541442841651026","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27186001621007168581611489439038285508","series":"1.2.826.0.1.3680043.8.498.84393876088808274463002503414958988504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27197413293500684424735285039923168926","series":"1.2.826.0.1.3680043.8.498.22052119037502445961862038295316639376","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545","series":"1.2.826.0.1.3680043.8.498.98634828695513736700311574500945447709","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355","series":"1.2.826.0.1.3680043.8.498.25832612746165816366913163286998811236","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27324284859290040148888098857880927353","series":"1.2.826.0.1.3680043.8.498.63971202124729391516261039039782889861","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27422742748182689233651649793019570511","series":"1.2.826.0.1.3680043.8.498.66584380164421571869437902401678075401","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27456622705361831116491505615162380604","series":"1.2.826.0.1.3680043.8.498.41457892961284245673169390278604488595","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27474440682441854303485403899867525864","series":"1.2.826.0.1.3680043.8.498.12546153241462180982793601369689398805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27644321991318957631114727895883331622","series":"1.2.826.0.1.3680043.8.498.82659397644781049874999326656262962181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27655410895168998898506695863000928400","series":"1.2.826.0.1.3680043.8.498.71437527401457581741993499501541792614","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.90183771466143440254584459521416778775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.56274873907983167341221101557298977463","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.62347154379215784513497657217587682026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316","series":"1.2.826.0.1.3680043.8.498.12191617424041831122024014080019632742","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27921111050680927418542588425335218896","series":"1.2.826.0.1.3680043.8.498.68283840078189246910438189795697027671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227","series":"1.2.826.0.1.3680043.8.498.46686954510155562007723966007041568232","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28011057282528460306257420499560926354","series":"1.2.826.0.1.3680043.8.498.83743655849565367650519982106676963410","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28040645186226259590182363324146810031","series":"1.2.826.0.1.3680043.8.498.54613335559232708658971338590896930089","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28132506470634542229651932141193521661","series":"1.2.826.0.1.3680043.8.498.12763002423889392233693174901251215954","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475","series":"1.2.826.0.1.3680043.8.498.13338309248899743480215029329519601599","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.94741796065001006160244007433885481283","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28289838818003822677994912735532050038","series":"1.2.826.0.1.3680043.8.498.66468758562523185533468123248213455817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306","series":"1.2.826.0.1.3680043.8.498.11242584930019313614216021355777811089","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28360174615866226538417948042981668998","series":"1.2.826.0.1.3680043.8.498.53274268993873912262853099401412140863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.10813360787614236517394609143558721500","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28539130184243787339297053654373783542","series":"1.2.826.0.1.3680043.8.498.10748757560261680520958283504172658774","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28626044336302232656987963623687293705","series":"1.2.826.0.1.3680043.8.498.13111246344971921803737272994903836178","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719","series":"1.2.826.0.1.3680043.8.498.19203937341329868873463115109021826382","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28646304373057947277669139986982208473","series":"1.2.826.0.1.3680043.8.498.93877510376348256209922440270441185037","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687","series":"1.2.826.0.1.3680043.8.498.99133438631356576124046130048556136901","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28770594655815108010532953376271504596","series":"1.2.826.0.1.3680043.8.498.12832253104992858486120348470492322235","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28848511016500781798837621261660844237","series":"1.2.826.0.1.3680043.8.498.21601749190552401886881557292694725043","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608","series":"1.2.826.0.1.3680043.8.498.94669021678379211035121750893811429094","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28946722216615442736212359468410891872","series":"1.2.826.0.1.3680043.8.498.57590514577223763321310593464320724533","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29039872266866623711060910233238440882","series":"1.2.826.0.1.3680043.8.498.93450319017267502077494549920227824765","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29090158364700006304493717187346344597","series":"1.2.826.0.1.3680043.8.498.12065886853379919198667511759378467059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181","series":"1.2.826.0.1.3680043.8.498.10950984403724175752275731316112182856","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665","series":"1.2.826.0.1.3680043.8.498.42857429402372416119411013083267335182","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595","series":"1.2.826.0.1.3680043.8.498.14617178890647606690274296175063145051","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29384016102624554484300773136383156153","series":"1.2.826.0.1.3680043.8.498.11167603516024917462663413306694526499","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878","series":"1.2.826.0.1.3680043.8.498.29927399140683157557450106379360457671","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856","series":"1.2.826.0.1.3680043.8.498.88635849433117067491018765625876948713","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569","series":"1.2.826.0.1.3680043.8.498.11664247862831881323395155411890212447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550","series":"1.2.826.0.1.3680043.8.498.10554209211078193916454348670841484936","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049","series":"1.2.826.0.1.3680043.8.498.54039394830712896003024761100530237257","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29971686675947075024168289004002445393","series":"1.2.826.0.1.3680043.8.498.13187856990960059279704742615377977376","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30066410959464367576375502522994033142","series":"1.2.826.0.1.3680043.8.498.10639953179853628925129417597150207758","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.12426348499030334412652815775824040433","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30174064884443577177838026250637902388","series":"1.2.826.0.1.3680043.8.498.40695102900909094536257341201355408391","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30222189977108980260768952691472020722","series":"1.2.826.0.1.3680043.8.498.13718563739224630576083911729727755037","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600","series":"1.2.826.0.1.3680043.8.498.15138103164892668700835246228249400720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098","series":"1.2.826.0.1.3680043.8.498.40266574990380322506254792835648208050","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30330001893437233719772129404417384472","series":"1.2.826.0.1.3680043.8.498.12229653712279685358116641420124270953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","series":"1.2.826.0.1.3680043.8.498.13084212301309701865523812476487503851","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.10611618947763489886507677940119186847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.79839955925555385862489786610386324420","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612","series":"1.2.826.0.1.3680043.8.498.23592879396890513601879143787220522863","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30672390215843894382866701432377109444","series":"1.2.826.0.1.3680043.8.498.97580862359175761294206107532310061586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30746167840159843733412534591988948270","series":"1.2.826.0.1.3680043.8.498.33231707118602307177817721945696602758","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.75393325317316365528253444802895350897","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374","series":"1.2.826.0.1.3680043.8.498.18998758027097646382710114175513785621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070","series":"1.2.826.0.1.3680043.8.498.11432563015399976786951923774678010218","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30942874102560162331831960649006411381","series":"1.2.826.0.1.3680043.8.498.27403545604174100563751898738434969237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30998941785989496591079894962766360976","series":"1.2.826.0.1.3680043.8.498.10722130763966226643727120662626974772","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31080452089421812812019413302222437434","series":"1.2.826.0.1.3680043.8.498.75473650725246715936629050077164015033","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31137162827930164367134811210741362416","series":"1.2.826.0.1.3680043.8.498.39637485763323390472123300144556825913","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669","series":"1.2.826.0.1.3680043.8.498.91576434277578423609361456096971088438","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","series":"1.2.826.0.1.3680043.8.498.21955096597342108247751465199911811390","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.44229325504236472149695083754122629863","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31282366761674650660824706600044115805","series":"1.2.826.0.1.3680043.8.498.12357413357932167669276365156027984640","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31352187408311685267245948214616940728","series":"1.2.826.0.1.3680043.8.498.10856569820141167252208977902501907943","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31461240229123701082444627629841504099","series":"1.2.826.0.1.3680043.8.498.68942275822504824749095981274004681772","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31565705236407722278921437231164204914","series":"1.2.826.0.1.3680043.8.498.85490391937539256259793954289185026797","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31640159014711750808214354395404624815","series":"1.2.826.0.1.3680043.8.498.61778802676670222566366204655295581769","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31792234576444017394182974951507838927","series":"1.2.826.0.1.3680043.8.498.42521366664406243984235972836390397785","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882","series":"1.2.826.0.1.3680043.8.498.11663456935307515342346667410140230433","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100","series":"1.2.826.0.1.3680043.8.498.10768066460711869560851798295748897948","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024","series":"1.2.826.0.1.3680043.8.498.69915568861624461197560004069634312000","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32078592787308498852570411454111970699","series":"1.2.826.0.1.3680043.8.498.78858565248652037496917243670201752316","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816","series":"1.2.826.0.1.3680043.8.498.50816920845531698674157273670020820692","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32226253011052584155674401943276961907","series":"1.2.826.0.1.3680043.8.498.50253959585692500999605462999378120072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631","series":"1.2.826.0.1.3680043.8.498.39430867712813130850983404781379010714","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32328699307307661181881949763360098439","series":"1.2.826.0.1.3680043.8.498.12406484342649922839077506433283914861","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654","series":"1.2.826.0.1.3680043.8.498.26908885173050592995580817495954270998","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32488933488399304214586978746578986823","series":"1.2.826.0.1.3680043.8.498.29193633516554644378719792003009074845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399","series":"1.2.826.0.1.3680043.8.498.39619707993850630953848715130755261018","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933","series":"1.2.826.0.1.3680043.8.498.78624689887451823512696130147616171519","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.71365747616259757513070467613637309056","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32704159300269838731722725240928883385","series":"1.2.826.0.1.3680043.8.498.11650560850326488893351000332961810087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32784816158214124982843838883515640569","series":"1.2.826.0.1.3680043.8.498.55031002334633842816474783723450394872","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753","series":"1.2.826.0.1.3680043.8.498.63651442567731832235546321563651661729","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32916544979513354644075485240094539251","series":"1.2.826.0.1.3680043.8.498.14485436598708225703602871950115484139","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33085434860684110795966817803985794300","series":"1.2.826.0.1.3680043.8.498.88637254104557478452337587859197561381","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.38776911808045893000989787544975618340","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33273874534046658504935877940537027702","series":"1.2.826.0.1.3680043.8.498.58469835620717960282667268734858517880","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33324670630108603340869986205298694836","series":"1.2.826.0.1.3680043.8.498.94974252317386233378698004304889406889","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33387430823462011959198830568803117633","series":"1.2.826.0.1.3680043.8.498.12706985314218579395874813071667893909","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684","series":"1.2.826.0.1.3680043.8.498.10621488047542633610438487352704341212","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521","series":"1.2.826.0.1.3680043.8.498.35697967323951117172955835449540732394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33620268340279901383196638228030589893","series":"1.2.826.0.1.3680043.8.498.10762249251520789297502030571384861436","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33707148112045883958228656491167647755","series":"1.2.826.0.1.3680043.8.498.87334914289394475155240230995543665431","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33834227140201599591317512187502720859","series":"1.2.826.0.1.3680043.8.498.10841805699218611762079818273077113637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650","series":"1.2.826.0.1.3680043.8.498.82434269122411648985185983074006460096","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33947148016115721563601115463364787679","series":"1.2.826.0.1.3680043.8.498.39865574089232925854148660407242172937","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33997019840409023169542161093833920630","series":"1.2.826.0.1.3680043.8.498.11386107710605058607928783880822631779","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34058295828142876023659425586740832405","series":"1.2.826.0.1.3680043.8.498.16442305065114460818362458984867773737","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514","series":"1.2.826.0.1.3680043.8.498.50408683918501383379340876847217619053","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34126843286927697018624309801437341727","series":"1.2.826.0.1.3680043.8.498.67507426011627402233008146716850853423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","series":"1.2.826.0.1.3680043.8.498.26601832261391565367775939111731206423","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613","series":"1.2.826.0.1.3680043.8.498.39321650168095989597368765336620087186","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34433343440016591348979084714300062954","series":"1.2.826.0.1.3680043.8.498.10517520196509105745861272651855351136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34501976934829436985900782482912019868","series":"1.2.826.0.1.3680043.8.498.89942652591543715726185813122520450500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444","series":"1.2.826.0.1.3680043.8.498.12445890653975852753922197340594162415","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906","series":"1.2.826.0.1.3680043.8.498.16179480387371023876536579131242485170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34685905030370793639196564723935583035","series":"1.2.826.0.1.3680043.8.498.56454085218863048442970283058276928820","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599","series":"1.2.826.0.1.3680043.8.498.11342811059308329081502316290218180230","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.12467618510936228784132528528498918173","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34826100703045042103752460066224521343","series":"1.2.826.0.1.3680043.8.498.64025193206719723130269571617108298647","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579","series":"1.2.826.0.1.3680043.8.498.58972222648973778170819220813499720770","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35017973257585430544005763350601015208","series":"1.2.826.0.1.3680043.8.498.37512987896029262638959030426017510751","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35027750475162395113025853006474122989","series":"1.2.826.0.1.3680043.8.498.63789494474050056283332363910790047594","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35121262802959257590413582180924083425","series":"1.2.826.0.1.3680043.8.498.12864594829349007718792343111589268991","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761","series":"1.2.826.0.1.3680043.8.498.10041991293677454742693078448112712633","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35257158249961136373696255071228322165","series":"1.2.826.0.1.3680043.8.498.66228769815319852831952139546881298208","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.44858952835471288494114935421227434310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35542843700847404584397532402313561503","series":"1.2.826.0.1.3680043.8.498.12742241430326195442424996966274249637","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35572306796137927247415883445612561056","series":"1.2.826.0.1.3680043.8.498.11192182402082894181596918418092194720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.13248300747862274862000577323022935809","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35717294165699019392845830077937768611","series":"1.2.826.0.1.3680043.8.498.10544680177099036736741990602132666289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944","series":"1.2.826.0.1.3680043.8.498.88008642774969788015939189027816886031","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35826402289277331555736683320175912112","series":"1.2.826.0.1.3680043.8.498.46746337493552861710391331927293484531","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201","series":"1.2.826.0.1.3680043.8.498.99739985381361627965067949832185359337","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314","series":"1.2.826.0.1.3680043.8.498.86471144737866460329054427352932157832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923183633813281697521537744486885300","series":"1.2.826.0.1.3680043.8.498.75348041335018144658615461391314543755","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35932368810339127867306228699757809449","series":"1.2.826.0.1.3680043.8.498.11099129926261415575017192066779532767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35961087303991442751747064990497975939","series":"1.2.826.0.1.3680043.8.498.12707370772357392150407770287366011055","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006","series":"1.2.826.0.1.3680043.8.498.21382607958338990345059293309321040150","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36083794505435625841225237415909011547","series":"1.2.826.0.1.3680043.8.498.28209543009687270149799493429570974472","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36131369236532190338276701919104598475","series":"1.2.826.0.1.3680043.8.498.13156006894964793586674452905449295705","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36185754164795053628650116141620545139","series":"1.2.826.0.1.3680043.8.498.51128772462985377915969105352040950496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.48105729583225438728323962532343308494","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.12020690197798945699452991496732363958","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36431753977367847587395456605556703040","series":"1.2.826.0.1.3680043.8.498.99106493093769424451322627700664523277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36488342368050295711098478501722217415","series":"1.2.826.0.1.3680043.8.498.59204176171437983524540040885674352886","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36599566036331923007586730435849419272","series":"1.2.826.0.1.3680043.8.498.11860862705108494914326101191906055177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.12837935739917159712035257892208218631","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985","series":"1.2.826.0.1.3680043.8.498.12015821021961440993253578002845032997","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100","series":"1.2.826.0.1.3680043.8.498.68040250872619961079537599761549347953","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36846391753332367835804699576399553980","series":"1.2.826.0.1.3680043.8.498.15078050270513747680260108971117505458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36924091633099385100942228071545017565","series":"1.2.826.0.1.3680043.8.498.11624019166784337431847252238738032935","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991","series":"1.2.826.0.1.3680043.8.498.12962549911879906038213918752154837541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876","series":"1.2.826.0.1.3680043.8.498.12938649190925176791668147819924702046","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868","series":"1.2.826.0.1.3680043.8.498.11606651026907404375395827153772720395","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.84350927740255633749568334887759860909","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895","series":"1.2.826.0.1.3680043.8.498.94208301767140589849083610851676702853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37347132459817985668038603215490355489","series":"1.2.826.0.1.3680043.8.498.26098404041292977939031647284137255862","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37390494423735156928337187351900345080","series":"1.2.826.0.1.3680043.8.498.99120600839394087811796277319933498197","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338","series":"1.2.826.0.1.3680043.8.498.12603448355065471945088383960619796642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37438631765770807892914198396852544668","series":"1.2.826.0.1.3680043.8.498.18860132229373683394189431129852609865","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37485075271886447709158414670394650122","series":"1.2.826.0.1.3680043.8.498.89628172404318549763130729041493410761","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37510478969810784660124747242051166554","series":"1.2.826.0.1.3680043.8.498.11920150760666792325355026220234966165","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023","series":"1.2.826.0.1.3680043.8.498.82105313840293074086548975907598870242","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.20293459293337315733171566852773780812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597","series":"1.2.826.0.1.3680043.8.498.22068887316070570188227062451016278149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37713008080610827760634302982073393454","series":"1.2.826.0.1.3680043.8.498.94605361190149919501047265496390593091","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680","series":"1.2.826.0.1.3680043.8.498.87813801633306918004512820977976658354","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.89561984729430965399670803292961875837","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37885316830093262977685802809269466252","series":"1.2.826.0.1.3680043.8.498.29985729870910295678456748777604425390","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37936061471366103885185528435327650981","series":"1.2.826.0.1.3680043.8.498.11701762891508090107346803314817269062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38010459551726691601986492657296948013","series":"1.2.826.0.1.3680043.8.498.10894026281107411051127819578933787962","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38056024157303576597111203925091929489","series":"1.2.826.0.1.3680043.8.498.22090691320480212878631977338382661215","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38085359363401170553191582038767900323","series":"1.2.826.0.1.3680043.8.498.94986430719771597291548677448374538173","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38173249623713271734596884672770674290","series":"1.2.826.0.1.3680043.8.498.84444206517437589581922367990417406499","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38233845434092134418342290639986261345","series":"1.2.826.0.1.3680043.8.498.73129675636393991104780056291936907234","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389","series":"1.2.826.0.1.3680043.8.498.32411939378392111882317957475593089387","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38306262614552497897681886368029860358","series":"1.2.826.0.1.3680043.8.498.22218528349487640325750070334236134440","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018","series":"1.2.826.0.1.3680043.8.498.68764078527811705714905544259191522305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.78420932581348415752928030840662649246","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38427680930662301114332563903957471851","series":"1.2.826.0.1.3680043.8.498.37391485278543206690377671600104314540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.42050206895589219752083423906349105349","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267","series":"1.2.826.0.1.3680043.8.498.39204462863930218814570124798572764076","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200","series":"1.2.826.0.1.3680043.8.498.44247127955857486447844376514845964808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451","series":"1.2.826.0.1.3680043.8.498.10884869147970337457812860207583878646","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736","series":"1.2.826.0.1.3680043.8.498.58191595288441834384948068226260200277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219","series":"1.2.826.0.1.3680043.8.498.12919729600620754093223274052869963429","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38774313460931748278644354956408578813","series":"1.2.826.0.1.3680043.8.498.65776958345212317317052981318627000440","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","series":"1.2.826.0.1.3680043.8.498.20795853606280876721847967908182191092","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060","series":"1.2.826.0.1.3680043.8.498.19531844795067361093265142439397329806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38900640030483599614088224967855884221","series":"1.2.826.0.1.3680043.8.498.80961740096143710156745873416293298445","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38928323954764175259962816410233336718","series":"1.2.826.0.1.3680043.8.498.19046096204969131734899141046235947659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38940868544480800333574786964232944756","series":"1.2.826.0.1.3680043.8.498.29905086090585880938661496899167014043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354","series":"1.2.826.0.1.3680043.8.498.11460434292147962660792903502775033671","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427","series":"1.2.826.0.1.3680043.8.498.38646250812466941913680469033941521301","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053","series":"1.2.826.0.1.3680043.8.498.30176221216136974946524188260048463109","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584","series":"1.2.826.0.1.3680043.8.498.77431198198668556284174156727516251297","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39264696726258404531295961801831089064","series":"1.2.826.0.1.3680043.8.498.97460077265085610316311160584601218736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709","series":"1.2.826.0.1.3680043.8.498.90984831969845653576494166313937093269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764","series":"1.2.826.0.1.3680043.8.498.11677656723190179504584054516364428954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39507184510584030713933330702725991027","series":"1.2.826.0.1.3680043.8.498.68384294127195734241053695728868762172","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912","series":"1.2.826.0.1.3680043.8.498.24502522665928850423336246381624105813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39642276506855996181597769215091644995","series":"1.2.826.0.1.3680043.8.498.10351437885345351588684572666029731296","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.12671936527487066109882935772951246331","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592","series":"1.2.826.0.1.3680043.8.498.65277253159507034215071718295610904186","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.65526307042725886629991487165971534833","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39969071491383654382076256213867635377","series":"1.2.826.0.1.3680043.8.498.32316150920123303470436518517685080804","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40041164441539690764994039690598305675","series":"1.2.826.0.1.3680043.8.498.12646364677057330927725112448030565630","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40154708689850542672289658249059660788","series":"1.2.826.0.1.3680043.8.498.43459181085703893987700532550393746243","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.12784973474686413928963162620840307787","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40257126480600927758308319831518876543","series":"1.2.826.0.1.3680043.8.498.84439267731350612782265677786317400302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230","series":"1.2.826.0.1.3680043.8.498.23514991462520560182899362526504554794","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.73058586175072828955978383828838092358","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679","series":"1.2.826.0.1.3680043.8.498.12114721488586862213375775948559145716","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40441879276927383122480339472406105572","series":"1.2.826.0.1.3680043.8.498.82830577215880165883617810083259840751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40547250250566982489293057895661861722","series":"1.2.826.0.1.3680043.8.498.38217903187938224939259771933309635959","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037","series":"1.2.826.0.1.3680043.8.498.22770241138907299163221718370803425341","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.28914566140164588975198377975783903254","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.44664696467444045309216389826259374933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476","series":"1.2.826.0.1.3680043.8.498.13039886767376496446735924820376092564","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40828786094337186697222449293183309391","series":"1.2.826.0.1.3680043.8.498.50617761408306205469257018995292929370","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40898220266193841898869905714214454387","series":"1.2.826.0.1.3680043.8.498.90087739576626914042992225696119075128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40930045208346676628855922755791787949","series":"1.2.826.0.1.3680043.8.498.38897059883511605467107105050359413017","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40994616246264625946610553907227975893","series":"1.2.826.0.1.3680043.8.498.82315896375170918690690370930258890930","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41108421820714108307078614688022098388","series":"1.2.826.0.1.3680043.8.498.17583678470660204634283680800497765133","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212","series":"1.2.826.0.1.3680043.8.498.13377310851697199841722756709274290201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","series":"1.2.826.0.1.3680043.8.498.78596650630081320539763268867378351054","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41326575785634778198143222323166756210","series":"1.2.826.0.1.3680043.8.498.31819962930866544028747728667057906952","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41382780862853765983216810855531499640","series":"1.2.826.0.1.3680043.8.498.76021278689335832357984464034841974009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41440206520769979754790425253631809212","series":"1.2.826.0.1.3680043.8.498.46127016288256458107230538124622381622","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41564229176277158115386256329157870722","series":"1.2.826.0.1.3680043.8.498.60849842447599033644401231175794770934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41605033773891953796694219242501252567","series":"1.2.826.0.1.3680043.8.498.55781995199407905899613220212897861820","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41623512571866040054960002151226127775","series":"1.2.826.0.1.3680043.8.498.87889855635979019625940166982641713033","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41662013720606493874198348776219875496","series":"1.2.826.0.1.3680043.8.498.42879818538697766866720622659495632510","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41696746845418651508748386392327996143","series":"1.2.826.0.1.3680043.8.498.12491740159371856900685765977643134024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41717437729988689273718049162153247404","series":"1.2.826.0.1.3680043.8.498.86571242478652766655850307967428062538","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535","series":"1.2.826.0.1.3680043.8.498.99266807330431804817491074980070351467","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683","series":"1.2.826.0.1.3680043.8.498.65041254621911892794407126627595246829","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41856137734822068230231773222820851271","series":"1.2.826.0.1.3680043.8.498.10366854109360278006241562622918042494","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480","series":"1.2.826.0.1.3680043.8.498.84498154609720668750354554323901627515","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808","series":"1.2.826.0.1.3680043.8.498.92460427929089307404194462660495341908","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725","series":"1.2.826.0.1.3680043.8.498.20740847771867953537505885946507272400","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42130173624033946465704266507948122088","series":"1.2.826.0.1.3680043.8.498.11412114188991364342152465178757843746","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42225165085995728538373532053227662192","series":"1.2.826.0.1.3680043.8.498.93058577172192560905219241960793101251","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633","series":"1.2.826.0.1.3680043.8.498.84367197989256405733825045764395168812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42414269167296543219685897036829241885","series":"1.2.826.0.1.3680043.8.498.81790093930212834379324926276780542518","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876","series":"1.2.826.0.1.3680043.8.498.12501374342274947844039970442650265523","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276","series":"1.2.826.0.1.3680043.8.498.12743186420169643703006675736659830078","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42609322631348207494943622999125822017","series":"1.2.826.0.1.3680043.8.498.23133684103394134756015942136675724734","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42719592405241698450757802131459694665","series":"1.2.826.0.1.3680043.8.498.12580239212753212961772056996393333065","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394","series":"1.2.826.0.1.3680043.8.498.10928512166853806405077996433444978202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214","series":"1.2.826.0.1.3680043.8.498.89838823994627868761100743419235250625","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.27041737837814703143257818370909997470","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42942844751304929338846069579289911684","series":"1.2.826.0.1.3680043.8.498.10028937422633444173012400353019117357","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744","series":"1.2.826.0.1.3680043.8.498.12783693695015789030453471765095541485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887","series":"1.2.826.0.1.3680043.8.498.62966321450173067339903490074889047075","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43127139928509033152155587499980767364","series":"1.2.826.0.1.3680043.8.498.13770144693198715321625592135087352788","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43256282639827216310036033378453939750","series":"1.2.826.0.1.3680043.8.498.27969495942995047255131560546207103868","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43325226037168674077736336919956196684","series":"1.2.826.0.1.3680043.8.498.12354462582203476285934765015430555304","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43351215447181735764667371462573550267","series":"1.2.826.0.1.3680043.8.498.30192361579134204392145694623495035097","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43432499267536761263202158662502957684","series":"1.2.826.0.1.3680043.8.498.91493326418980665649220316637174255245","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43442933855807685981061557274166869461","series":"1.2.826.0.1.3680043.8.498.97961602282474400736733509132026458734","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43587191909319947256183198986671106743","series":"1.2.826.0.1.3680043.8.498.29549024700343261774682452897657319319","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163","series":"1.2.826.0.1.3680043.8.498.32403898021975919777298213680458676063","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.43797037142729611453626414920554246615","series":"1.2.826.0.1.3680043.8.498.59154629924206995540776323861601373663","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43967752580489865323542041346927244370","series":"1.2.826.0.1.3680043.8.498.11250269653635349864572039008259600900","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.49622014172976904703418228614287939737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44259687212868598275713310246316207497","series":"1.2.826.0.1.3680043.8.498.14908519531683702841551261327038130859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44323131330689365651417881352766725366","series":"1.2.826.0.1.3680043.8.498.15460965319268552705030997659943718302","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44365814642869065241166313231424697474","series":"1.2.826.0.1.3680043.8.498.87302230651707437387509021118343603850","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44417768181231991989438194369583907493","series":"1.2.826.0.1.3680043.8.498.11565575631477282195450986421719292762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713","series":"1.2.826.0.1.3680043.8.498.51214921587197073107742526262641733786","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.12834652472914211438729881471532713053","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44670636848627131733157841382853275817","series":"1.2.826.0.1.3680043.8.498.63623212065038299734930944150141057032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803","series":"1.2.826.0.1.3680043.8.498.10966378929391113067204923450061714601","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44863396093454506630473164990623458425","series":"1.2.826.0.1.3680043.8.498.43718193914165285997277605418816690057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025","series":"1.2.826.0.1.3680043.8.498.26543928649623342796324443695003832198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","series":"1.2.826.0.1.3680043.8.498.11810368109524245394040516454707362097","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45033326048521572717771708818514772861","series":"1.2.826.0.1.3680043.8.498.47415404664182899263803549378674925712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45082393649917309927267478072569185694","series":"1.2.826.0.1.3680043.8.498.17788328184588251927885322950125517044","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45170517567841608067443743961197123093","series":"1.2.826.0.1.3680043.8.498.13310056883669203243232669647401585738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632","series":"1.2.826.0.1.3680043.8.498.51873372323793318811325897752737944800","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45304570280714567513866386380356988876","series":"1.2.826.0.1.3680043.8.498.16119541353240457512098376060532796704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959","series":"1.2.826.0.1.3680043.8.498.98762648805757038024428385073135471220","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45368532788186860828867663606651818835","series":"1.2.826.0.1.3680043.8.498.75451190201850085135782013701481092364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45406212529450909512649737062964080190","series":"1.2.826.0.1.3680043.8.498.89012702049780032197195847714513835306","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45463321406810606039579747001339703552","series":"1.2.826.0.1.3680043.8.498.66293896342342687697153793036008267769","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45504348071514081636909771148066956410","series":"1.2.826.0.1.3680043.8.498.10750154392894150734928372969045961849","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642","series":"1.2.826.0.1.3680043.8.498.13398048817436588629564251223949438038","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674","series":"1.2.826.0.1.3680043.8.498.23828495233746103725927389344211870827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45811964949115542204306831813385709206","series":"1.2.826.0.1.3680043.8.498.11259837677802686840326653002836878115","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45865541491386404818767748310365919685","series":"1.2.826.0.1.3680043.8.498.97822902823042260030610084651944252656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614","series":"1.2.826.0.1.3680043.8.498.31348287605696677946704651618605568032","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397","series":"1.2.826.0.1.3680043.8.498.15716097694974347844771051995284419043","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46052469424267861803032899866428547156","series":"1.2.826.0.1.3680043.8.498.91857637431593712338535283548414069461","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46222691720198967161306520058359099856","series":"1.2.826.0.1.3680043.8.498.12686227330494485765002247609021474738","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46259494895672269475940484605821837768","series":"1.2.826.0.1.3680043.8.498.76431557930359087988469840713548913991","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523","series":"1.2.826.0.1.3680043.8.498.43550919479488263545697405289931263001","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46309540159823359540686782651264853894","series":"1.2.826.0.1.3680043.8.498.11878188644382793527107726192679425870","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766","series":"1.2.826.0.1.3680043.8.498.81358212941052617035277035145128506817","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46506161284114284997220751358509505380","series":"1.2.826.0.1.3680043.8.498.11626421817563054861468066665954796003","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479","series":"1.2.826.0.1.3680043.8.498.87677123519034406084600221799247472759","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552","series":"1.2.826.0.1.3680043.8.498.27100211009487313627553228568735004485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639","series":"1.2.826.0.1.3680043.8.498.44959805013741338361863552098125511193","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46842734152337184482152392783260161845","series":"1.2.826.0.1.3680043.8.498.75122617106428603012817129835141101775","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46893956584698923808118623450380169212","series":"1.2.826.0.1.3680043.8.498.99736918927114303702478142458921549308","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.77942974038672717908988987061368633683","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029","series":"1.2.826.0.1.3680043.8.498.41597838660454579397036006576394373506","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47080537724499166054609508776778189902","series":"1.2.826.0.1.3680043.8.498.21468559949741969433848975212433193580","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47239511009810375185594384849101241004","series":"1.2.826.0.1.3680043.8.498.71653599112502691025936023641556330330","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642","series":"1.2.826.0.1.3680043.8.498.46048398970353203297472927557500056088","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955","series":"1.2.826.0.1.3680043.8.498.10715763686515083755927290255558669571","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684","series":"1.2.826.0.1.3680043.8.498.11916006522500570183747265982000158430","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246","series":"1.2.826.0.1.3680043.8.498.58181932119981739577966729315913991450","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47655269404919497034805209368250028981","series":"1.2.826.0.1.3680043.8.498.12397566320943383265870338603540082635","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710","series":"1.2.826.0.1.3680043.8.498.11157119427745919051681014210005093722","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174","series":"1.2.826.0.1.3680043.8.498.14480654913889023737031021400976479211","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47870074144891988520855535250618228390","series":"1.2.826.0.1.3680043.8.498.21804286337019509271793121226932960757","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552","series":"1.2.826.0.1.3680043.8.498.42897217769815591878922010519102067733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541","series":"1.2.826.0.1.3680043.8.498.59802343928383391921450135737418059667","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48308485987218296338173512060246238691","series":"1.2.826.0.1.3680043.8.498.12533222555248724891489135920386135063","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239","series":"1.2.826.0.1.3680043.8.498.12326610874209177111674275790891789804","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776","series":"1.2.826.0.1.3680043.8.498.98002967147635676538343260118227459731","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236","series":"1.2.826.0.1.3680043.8.498.12644487516604648225924740249540260193","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48564780460694033783288540237012068889","series":"1.2.826.0.1.3680043.8.498.84195493448908831588588934451927423932","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545","series":"1.2.826.0.1.3680043.8.498.30068766523662956738633036004535664927","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244","series":"1.2.826.0.1.3680043.8.498.11827182563340099784785171494407395081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48818931767017230678539144135307491171","series":"1.2.826.0.1.3680043.8.498.89743232203268345532079159153504018234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.22711502467844567048975634006083779337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48933020429865163393792899740045803587","series":"1.2.826.0.1.3680043.8.498.57949976739673130184888944035938144874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49025249926859976858200443939208207124","series":"1.2.826.0.1.3680043.8.498.93326866370765706527570952673239009170","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262","series":"1.2.826.0.1.3680043.8.498.11629853075179398674953974303770214695","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49094276335794811195117175603556929507","series":"1.2.826.0.1.3680043.8.498.92662854344136928951201455739598296704","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49219586198038249030222134813685536295","series":"1.2.826.0.1.3680043.8.498.60335097231608359973576536951575817892","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49347024975371539869871751159907613638","series":"1.2.826.0.1.3680043.8.498.55933986742404825225538250280057292036","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330","series":"1.2.826.0.1.3680043.8.498.39652939265737398135529810882356347009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49481811111224238761880576486796603139","series":"1.2.826.0.1.3680043.8.498.12637267547506222342526855367559021144","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49495644703047055149965226028097259207","series":"1.2.826.0.1.3680043.8.498.70496800664874258458925055435998463049","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513","series":"1.2.826.0.1.3680043.8.498.12813923412635865441373705033824248639","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49660659440268002826070581802625582554","series":"1.2.826.0.1.3680043.8.498.91404408332153299904219484031387665689","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49703744074809050221928706774732375079","series":"1.2.826.0.1.3680043.8.498.47528272842021679768905677224521930202","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49850672111501136862025523789202033809","series":"1.2.826.0.1.3680043.8.498.66745646990240366989028234648524344369","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657","series":"1.2.826.0.1.3680043.8.498.66250585727951274254158656452575435481","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49887758195164927372912584780941789021","series":"1.2.826.0.1.3680043.8.498.94205760450940194983103011973899021059","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49942556484410764813681696157580687854","series":"1.2.826.0.1.3680043.8.498.23852129302756023082931152238245602539","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.51808728974227765593087975189165051481","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50089015780087008332657793589955395952","series":"1.2.826.0.1.3680043.8.498.68615986224017810815998653682622502224","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50144715912379960437506930683683218333","series":"1.2.826.0.1.3680043.8.498.11850555796855093205710745515364408910","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50183004658202912273462671442829139110","series":"1.2.826.0.1.3680043.8.498.12475693735560887459965418291785109981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50248717336781045512984550029203786086","series":"1.2.826.0.1.3680043.8.498.12618533659788697569268134592522883107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50287512028986928372350279452197816661","series":"1.2.826.0.1.3680043.8.498.66089669153624982688357286699958722864","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50342726517205151681842826322659275894","series":"1.2.826.0.1.3680043.8.498.43808573417258018894358191268931199766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.75202548546642260588324578474628427720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395","series":"1.2.826.0.1.3680043.8.498.59806679379512614043223568079316986187","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.10200060645286054663301728558173793668","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50666048236278074641860077012751071758","series":"1.2.826.0.1.3680043.8.498.95500496670269849890981752046260012268","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306","series":"1.2.826.0.1.3680043.8.498.12596471916835297358890038444477708563","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50694733929204626284366818623993960684","series":"1.2.826.0.1.3680043.8.498.99790198615121742953703282060485576145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130","series":"1.2.826.0.1.3680043.8.498.24911400544880153768071907432812682345","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50813123035221058777203670043892828582","series":"1.2.826.0.1.3680043.8.498.84106794595630268763160906078641543164","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784","series":"1.2.826.0.1.3680043.8.498.12599913938282165902401840019682785301","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845","series":"1.2.826.0.1.3680043.8.498.22676793355053247286208769596182092881","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50969691234820675847736765735133138324","series":"1.2.826.0.1.3680043.8.498.10429273640112515100778350866960903868","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51006813205605862503064447831381009425","series":"1.2.826.0.1.3680043.8.498.10897207461690182295539244735687429549","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102","series":"1.2.826.0.1.3680043.8.498.68689669295292281099665468121178507218","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51179760508587500025973916664101120069","series":"1.2.826.0.1.3680043.8.498.10798931103487775387119405266798889791","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51351386796317851725461803790273870995","series":"1.2.826.0.1.3680043.8.498.28971218055109347422046478087292202754","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51426696939581135481763379246368970360","series":"1.2.826.0.1.3680043.8.498.79348375880491479425805508336051581420","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51469564445249084310991187819410005207","series":"1.2.826.0.1.3680043.8.498.13751445552676572798077777234117925807","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116","series":"1.2.826.0.1.3680043.8.498.17204312796144440897970580207484904713","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257","series":"1.2.826.0.1.3680043.8.498.57517222056721688334169751709964142725","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51770131897828080663537020812721042704","series":"1.2.826.0.1.3680043.8.498.10089310559280531459496945123439716143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51809092313496476873962641559638828608","series":"1.2.826.0.1.3680043.8.498.18601672460275233476467812043341838793","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51828180600648126726057100018010765613","series":"1.2.826.0.1.3680043.8.498.11110280190794074816870685652896256970","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601","series":"1.2.826.0.1.3680043.8.498.68641382872294135279858562392424377758","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51917043703891091519740084517167861965","series":"1.2.826.0.1.3680043.8.498.35199134805747147070367637903687356473","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51981019698056120061499970572334688555","series":"1.2.826.0.1.3680043.8.498.90841348732209670916011564546763138295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52010826440192227660457107121818389761","series":"1.2.826.0.1.3680043.8.498.35818679767344676346107154106436486286","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52059522418267239652374290810994711032","series":"1.2.826.0.1.3680043.8.498.58675891289035922454691478103039840759","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.12416482268219306644386675076879779456","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52138967568492774495485454357106269286","series":"1.2.826.0.1.3680043.8.498.13126535071319247896542766119478301100","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52204390785696944340286127542242341202","series":"1.2.826.0.1.3680043.8.498.19009185466537653208679097977983992846","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065","series":"1.2.826.0.1.3680043.8.498.34910717820442203810921493538030416119","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52435631723314823200374985065756182593","series":"1.2.826.0.1.3680043.8.498.75754076204094506907182836586154087595","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52475562989926669452280424331791927661","series":"1.2.826.0.1.3680043.8.498.55697535355680814824327632319242427422","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52520288400853832269613213204540385687","series":"1.2.826.0.1.3680043.8.498.11031773691232018380342119758808227017","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52636981391578580436180193211113340410","series":"1.2.826.0.1.3680043.8.498.19181962490006073254156353597069568094","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535","series":"1.2.826.0.1.3680043.8.498.10433063857589847102604929200580755236","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52846023545870016653608198538437574931","series":"1.2.826.0.1.3680043.8.498.63212566007152972582963064478704983125","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52849154011665864497632297320961015159","series":"1.2.826.0.1.3680043.8.498.10819715869536630944699043768311196336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53025489296610571044246428509154950855","series":"1.2.826.0.1.3680043.8.498.12488663346238643077505670979147495144","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53080172184945962593713560671746700445","series":"1.2.826.0.1.3680043.8.498.55111271408731627635349802242270897311","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53114802640679819441521055029024037716","series":"1.2.826.0.1.3680043.8.498.99443560322258010462035409941255569419","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832","series":"1.2.826.0.1.3680043.8.498.21973256924729948936083383644075213228","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146","series":"1.2.826.0.1.3680043.8.498.11989968352167300691974272635907635981","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146","series":"1.2.826.0.1.3680043.8.498.55361082686815836481833699128493681355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53337684515627962545396495541158882262","series":"1.2.826.0.1.3680043.8.498.11636344058913130477537424604541239422","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263","series":"1.2.826.0.1.3680043.8.498.59473177994036969238765143676618961994","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53393741562201846141435914434030220216","series":"1.2.826.0.1.3680043.8.498.11982481054204774343749919511671208513","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.36145710216257536190943014592207935896","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53587920256280027683523152568832253423","series":"1.2.826.0.1.3680043.8.498.57359392330925735611305756827974079930","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53649895712172408476570770924813501118","series":"1.2.826.0.1.3680043.8.498.61311869154323740886641131873364001944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53792159922324410857755054939229546555","series":"1.2.826.0.1.3680043.8.498.65461086794174890322059295456439891443","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53893586200869613140666806653041177558","series":"1.2.826.0.1.3680043.8.498.11335585374631648853516480914556681600","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53963759852017695930808272438606800422","series":"1.2.826.0.1.3680043.8.498.94720470210106858743232038294258633410","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","series":"1.2.826.0.1.3680043.8.498.62318381531246889528400233360863086364","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763","series":"1.2.826.0.1.3680043.8.498.72781641234111085946942218587234767734","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54132814503202891434380002378663763655","series":"1.2.826.0.1.3680043.8.498.63806538023985092927484294044422795983","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54164004416963296439322252627774426375","series":"1.2.826.0.1.3680043.8.498.43503405747217821235451496138692543841","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54270039578867689669466339980682684126","series":"1.2.826.0.1.3680043.8.498.16088537310685163077290193301237919690","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.72267086364052333440899680309602663844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985","series":"1.2.826.0.1.3680043.8.498.13055884285956160513461590633630305108","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54427039222108917725902677899081931510","series":"1.2.826.0.1.3680043.8.498.75390850793130246399522147558085343358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54482848766921329667469114743547203929","series":"1.2.826.0.1.3680043.8.498.41845437902929449485042445000886859120","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54539703415639765348239790227512479583","series":"1.2.826.0.1.3680043.8.498.62638452789666005159491296530487567979","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589","series":"1.2.826.0.1.3680043.8.498.96715898920489697186592449700015630433","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557","series":"1.2.826.0.1.3680043.8.498.83255167036451047380807424342946827529","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54699196872730350880681018674893216245","series":"1.2.826.0.1.3680043.8.498.78355229406720743461008964261130495198","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.12682586727961470619891373308709951143","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54833384793167149128371137537527194757","series":"1.2.826.0.1.3680043.8.498.73925887062311070740758536123965441008","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450","series":"1.2.826.0.1.3680043.8.498.71042574996097068332053264058857876735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406","series":"1.2.826.0.1.3680043.8.498.16145768011031646130849076144471852866","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55065105486648830377200570585789726800","series":"1.2.826.0.1.3680043.8.498.24603382767187312184777903055916095392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55183153701332792661581741475876132309","series":"1.2.826.0.1.3680043.8.498.11284533805821290944782090508257637320","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593","series":"1.2.826.0.1.3680043.8.498.72958460580784894984309449111587233340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587","series":"1.2.826.0.1.3680043.8.498.28086463644967446841500718815662252605","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55397602116058381866334455456729470963","series":"1.2.826.0.1.3680043.8.498.13748681839649771437058470808590397034","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55583505147281684100202399148099429721","series":"1.2.826.0.1.3680043.8.498.10891593683517112996296489359971890421","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55653770127824574123236841437998533498","series":"1.2.826.0.1.3680043.8.498.12468390277939284902684474947679982763","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.12251347517495911147444134353813283071","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.28231384208661622954034581654261648232","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728","series":"1.2.826.0.1.3680043.8.498.19406937570547000087591260419888354060","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55887747667305814945719986974538814207","series":"1.2.826.0.1.3680043.8.498.10916385313127228522032099990597635790","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55930031395651221279170435293900769500","series":"1.2.826.0.1.3680043.8.498.13056369893806841921255434508276035674","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.22945670902061388647366647996556986977","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.93309182802959156307540810414620696944","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.28189672067719664601715768200522941923","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56156923369480999484275191793049573490","series":"1.2.826.0.1.3680043.8.498.70116325603570971640107814949398871057","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849","series":"1.2.826.0.1.3680043.8.498.58210238124682083984122566177056984379","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56269695574661037525483516531194514456","series":"1.2.826.0.1.3680043.8.498.44404098354996119346728666569497646833","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56385473079196655120993854770623075553","series":"1.2.826.0.1.3680043.8.498.11611380071666893601563322094429653848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56497268548664917337130798701015887461","series":"1.2.826.0.1.3680043.8.498.23488768607148213107214746541809111310","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.56357361530789913985100714173720169534","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56575942633137496743464648679775042979","series":"1.2.826.0.1.3680043.8.498.88567842149644355134953474286308840439","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006","series":"1.2.826.0.1.3680043.8.498.26406603940966961850060023141520095717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860","series":"1.2.826.0.1.3680043.8.498.78429725240667618078541682369685263914","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56796622077427019119540733986070534126","series":"1.2.826.0.1.3680043.8.498.53404842845227960875554584164370760344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445","series":"1.2.826.0.1.3680043.8.498.13786886845166051968125886233911349060","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56882015310689536014061315110969323429","series":"1.2.826.0.1.3680043.8.498.44128429354954482613507041984206303477","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56915079210789596803972894081756325026","series":"1.2.826.0.1.3680043.8.498.11186610401612292574647587059121238689","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56945039212126810522718038626526579591","series":"1.2.826.0.1.3680043.8.498.21740404798181847738990879402029840852","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56993108275828399240508695615212151102","series":"1.2.826.0.1.3680043.8.498.11866429136402723436901286972730756468","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.70001139436039083990536208820913263180","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856","series":"1.2.826.0.1.3680043.8.498.10523483176339155337759094026640447713","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57239450717434070615919278253974598533","series":"1.2.826.0.1.3680043.8.498.71328546318425465992418162732378072438","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475","series":"1.2.826.0.1.3680043.8.498.14204461412548002976720644165975534777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597","series":"1.2.826.0.1.3680043.8.498.24830807770099929255542105823013884340","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57703608122587911055931213603182679454","series":"1.2.826.0.1.3680043.8.498.86009796491912521311041037479028121365","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57775389384191131322519457853517177210","series":"1.2.826.0.1.3680043.8.498.12001697538275776524114685590406944795","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57811199787891897795884073594875380873","series":"1.2.826.0.1.3680043.8.498.80744703877388448716121106212091352808","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.13328190172337603116706563647720907424","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57882532838957577543516026386452055977","series":"1.2.826.0.1.3680043.8.498.84630305096293685237397054819737772992","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57951268100539350409054562078476694936","series":"1.2.826.0.1.3680043.8.498.97138242210208512581389187069420737883","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58051085126748057986491277367254533832","series":"1.2.826.0.1.3680043.8.498.10942412676200666443796707933705370164","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58250779958866595505585700517927225230","series":"1.2.826.0.1.3680043.8.498.19247869399882732240753599346888684727","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.11649034727065704727292055554275744388","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58346459085553680879151588730895136332","series":"1.2.826.0.1.3680043.8.498.27135538935088415225071932914494272707","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58404766256608483217059706069436050741","series":"1.2.826.0.1.3680043.8.498.45941617509491597232113177471456044948","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.26519712675749212244979136690926091679","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58482779112389764165599701941310919660","series":"1.2.826.0.1.3680043.8.498.57811601768163097267680186536018093748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167","series":"1.2.826.0.1.3680043.8.498.14184211466059223201272763706491709502","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101","series":"1.2.826.0.1.3680043.8.498.11598555476678968327347656393643743025","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656","series":"1.2.826.0.1.3680043.8.498.10906297317746561503197655749657706595","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58728013757221328590898166513995249334","series":"1.2.826.0.1.3680043.8.498.15266936718114844854269085675099770015","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130","series":"1.2.826.0.1.3680043.8.498.33928286104061894326820758207659912098","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538","series":"1.2.826.0.1.3680043.8.498.26971707487782974702636969029903999660","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58857477075095368361468658035369975572","series":"1.2.826.0.1.3680043.8.498.66593193627928860702488211761251700043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58925843611945635442423587804169242644","series":"1.2.826.0.1.3680043.8.498.11717174178604693602821153156660240103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59021913092219789232890327780064145948","series":"1.2.826.0.1.3680043.8.498.52814722371678159620722888660893320092","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305","series":"1.2.826.0.1.3680043.8.498.31536427483747905699749793661796069207","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768","series":"1.2.826.0.1.3680043.8.498.17237057261220953753721657674451593682","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59262240088132401543519466539885561578","series":"1.2.826.0.1.3680043.8.498.58104910196260256331086977797440612674","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823","series":"1.2.826.0.1.3680043.8.498.98906170029143795096480880895568091040","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59449702487184611096113620550399616423","series":"1.2.826.0.1.3680043.8.498.15200985599690951150902919578862050864","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710","series":"1.2.826.0.1.3680043.8.498.42984443688705446568460402049197716512","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59527896591755229337596505685198957637","series":"1.2.826.0.1.3680043.8.498.10823647123457650324015139368706970328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651","series":"1.2.826.0.1.3680043.8.498.53654964429428196873273423035178719399","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.73586374324306968107469048229525512801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59835677513721120155185666132462195647","series":"1.2.826.0.1.3680043.8.498.39684000108487952655679573975805706202","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59923507410103964137819195926587517759","series":"1.2.826.0.1.3680043.8.498.31009865598281531623146162045603179265","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59938475193950583662125713924156951751","series":"1.2.826.0.1.3680043.8.498.76023056879550289975367782135123219541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","series":"1.2.826.0.1.3680043.8.498.14854523419315257757435225685753545488","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.63000861685769444367937811934795700703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484","series":"1.2.826.0.1.3680043.8.498.48041494380797778614949854362512909815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60227430483444032531479844365716485643","series":"1.2.826.0.1.3680043.8.498.12539581961419842758833680744328830071","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60259828181878278590614805828898992241","series":"1.2.826.0.1.3680043.8.498.70538157703514937956523260363544558504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60394858470917657607846893765818799720","series":"1.2.826.0.1.3680043.8.498.30724179379883472168670402335913847490","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796","series":"1.2.826.0.1.3680043.8.498.29466981015102610707637205994706610655","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.60465293103574559546982440125572490190","series":"1.2.826.0.1.3680043.8.498.74892718414136063781126305384338152710","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60477027832057450004004942199870813886","series":"1.2.826.0.1.3680043.8.498.39541063372463149180714301021785427798","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60582762894748063612925320003677861384","series":"1.2.826.0.1.3680043.8.498.17591424111597820981690281624270717985","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331","series":"1.2.826.0.1.3680043.8.498.10047997472951043536409926572347675423","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60661676782543347731000874708337132276","series":"1.2.826.0.1.3680043.8.498.79440727097090710561312035866280336824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60840220723824894619895264673401129352","series":"1.2.826.0.1.3680043.8.498.18280941233170647898446428578044965659","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60931716414402855657852057585917605786","series":"1.2.826.0.1.3680043.8.498.65841185427729253304347742369091464205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558","series":"1.2.826.0.1.3680043.8.498.12078816155855116713675316968009351930","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61140619849833544411463750382539885141","series":"1.2.826.0.1.3680043.8.498.99775423944799325036101933152542684303","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169","series":"1.2.826.0.1.3680043.8.498.73134083170249017428901714517783505077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675","series":"1.2.826.0.1.3680043.8.498.11141168191908808060323667709940491129","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563","series":"1.2.826.0.1.3680043.8.498.11881502861260642219922855158748364319","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61435096069921474761812731150911059491","series":"1.2.826.0.1.3680043.8.498.42870857874934726387114190387981227634","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.37344621342987770292489738193971216251","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386","series":"1.2.826.0.1.3680043.8.498.92630095936244869619930951878470567735","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015","series":"1.2.826.0.1.3680043.8.498.58848273563927895425324779708468508580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61797111379749329977228618245440305623","series":"1.2.826.0.1.3680043.8.498.18871563063268621430659817662501499053","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61864939963243675140060416958055914981","series":"1.2.826.0.1.3680043.8.498.12131994528306531949403446647285463262","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569","series":"1.2.826.0.1.3680043.8.498.17734253084415029583618464171214058322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893","series":"1.2.826.0.1.3680043.8.498.72015515817970790437360033062987259774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62154969360158054310282857093665911261","series":"1.2.826.0.1.3680043.8.498.36031349134975089101421638685636133880","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775","series":"1.2.826.0.1.3680043.8.498.12083563555020925710772857217638702763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.11724042374686502907095153805684107613","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62329283530383340787096585492217632179","series":"1.2.826.0.1.3680043.8.498.34650981089239656245008234161742554232","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62400625562782184774973349983498962743","series":"1.2.826.0.1.3680043.8.498.87462433177842851430867176237448260282","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423","series":"1.2.826.0.1.3680043.8.498.92438117308951147868033669447393894115","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236","series":"1.2.826.0.1.3680043.8.498.64064664993094542843040674366148501286","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983","series":"1.2.826.0.1.3680043.8.498.13390183253914535805017733700313418028","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62776347582771469289738234607517546504","series":"1.2.826.0.1.3680043.8.498.69828899138568383191528228486444232224","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62830031603985608774418534497520663269","series":"1.2.826.0.1.3680043.8.498.79544725457045386451690787340403677852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","series":"1.2.826.0.1.3680043.8.498.10893968506378528215698739414225098125","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807","series":"1.2.826.0.1.3680043.8.498.10844087348354702494275597863077836876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63068687840516335034631353955761036465","series":"1.2.826.0.1.3680043.8.498.11362561076561854851194467130806753629","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63096952391833769395204944538276970114","series":"1.2.826.0.1.3680043.8.498.48107437754133795650969258208753249498","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316","series":"1.2.826.0.1.3680043.8.498.24593307024714701964697112334745815293","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360","series":"1.2.826.0.1.3680043.8.498.11860675779485900003061510778531494449","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63223220827342079068491460847713457238","series":"1.2.826.0.1.3680043.8.498.11669096688642970573364769002885566789","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63242029756172180508167933519465902827","series":"1.2.826.0.1.3680043.8.498.12020428752973228801827395954367561198","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895","series":"1.2.826.0.1.3680043.8.498.42042521077111408548009290480910952226","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519","series":"1.2.826.0.1.3680043.8.498.11728943395284353970868920073071957565","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63439188800533762543950349022148547855","series":"1.2.826.0.1.3680043.8.498.33000587225332500962313230756896239714","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63519003265636561965405570006747767668","series":"1.2.826.0.1.3680043.8.498.81495799721347481449078345675675735003","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63521726454879203860288713455843702564","series":"1.2.826.0.1.3680043.8.498.77769413682205982448586291066563195825","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63556690547679257097888724096967682731","series":"1.2.826.0.1.3680043.8.498.26698116722941482455389002673613450380","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258","series":"1.2.826.0.1.3680043.8.498.87284267896487979513580658748093193292","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809","series":"1.2.826.0.1.3680043.8.498.80449234366614740287298123956574901809","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334","series":"1.2.826.0.1.3680043.8.498.53537787516852481861835235924692752104","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566","series":"1.2.826.0.1.3680043.8.498.89691220902679176533980010631837587079","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64022112049842890696609934970222024490","series":"1.2.826.0.1.3680043.8.498.70745973073002788221936642810060809144","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567","series":"1.2.826.0.1.3680043.8.498.13192245180019854951176487428567683993","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64109231850522212435708176045583779359","series":"1.2.826.0.1.3680043.8.498.27707079228780823513779665330708100926","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373","series":"1.2.826.0.1.3680043.8.498.40871392656846845081743447452141024071","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64283618749539713756220539395944025878","series":"1.2.826.0.1.3680043.8.498.13393672233063471019247009179033015677","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64381606512330581256470279413868645789","series":"1.2.826.0.1.3680043.8.498.57128139282798557225269936832371902400","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910","series":"1.2.826.0.1.3680043.8.498.40051854759647629267374762012274626762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105","series":"1.2.826.0.1.3680043.8.498.55408118975064289061927017966742117292","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010","series":"1.2.826.0.1.3680043.8.498.43242959554077794502466311606370045206","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64606794953061143951972249525094490665","series":"1.2.826.0.1.3680043.8.498.12224677378072588408458348013457861832","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64652085172505930317660036583152917472","series":"1.2.826.0.1.3680043.8.498.36109601059677201324325838963111595855","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039","series":"1.2.826.0.1.3680043.8.498.61714171918691444339892175338123390353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64897285598346101578442697515294908341","series":"1.2.826.0.1.3680043.8.498.10048698899828841200257333243599606941","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525","series":"1.2.826.0.1.3680043.8.498.42996426682579964313562627579248394191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643","series":"1.2.826.0.1.3680043.8.498.11834876061248800329822935614773457837","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252","series":"1.2.826.0.1.3680043.8.498.29780894991873964576021375327681120998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455","series":"1.2.826.0.1.3680043.8.498.15336769469133173575277443505071144575","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65224808465738083245406235000047993647","series":"1.2.826.0.1.3680043.8.498.10013636672004583626703436913896976293","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816","series":"1.2.826.0.1.3680043.8.498.79042684809373367705982023567486345673","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65403984120528711726317048984986599537","series":"1.2.826.0.1.3680043.8.498.10763154974699255199880066457296813150","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719","series":"1.2.826.0.1.3680043.8.498.15369731391299152446177842031618019173","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65543472488214519069910341591668399186","series":"1.2.826.0.1.3680043.8.498.24902717187682773408092756765946695218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847","series":"1.2.826.0.1.3680043.8.498.42649460841179070412920579302679857130","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327","series":"1.2.826.0.1.3680043.8.498.71557939123642455232655031941705135396","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765","series":"1.2.826.0.1.3680043.8.498.36674507482464081910966636361272317808","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666","series":"1.2.826.0.1.3680043.8.498.55036766442151372168230621150068993941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674","series":"1.2.826.0.1.3680043.8.498.25546126387239152248368552514368119898","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.36287173784404727437618124509892547116","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980","series":"1.2.826.0.1.3680043.8.498.29225288386615380982931639554074646362","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66020213859389864645118554257523478921","series":"1.2.826.0.1.3680043.8.498.10689154928846578366308602599982808846","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398","series":"1.2.826.0.1.3680043.8.498.98245099402587417357529873580269242442","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887","series":"1.2.826.0.1.3680043.8.498.28991024359159631064423794774024587783","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66199248385068685539872840826351152652","series":"1.2.826.0.1.3680043.8.498.37003961743486455118514513112661445006","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084","series":"1.2.826.0.1.3680043.8.498.84742141127174967885668495940095965304","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177","series":"1.2.826.0.1.3680043.8.498.69850412482531731129305863633226404808","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66327751482076404294387513492434577338","series":"1.2.826.0.1.3680043.8.498.39172583107497604622773351352382154305","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421","series":"1.2.826.0.1.3680043.8.498.59693020212728361238654811826396248225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66489634638847834550694493585147280754","series":"1.2.826.0.1.3680043.8.498.27687384959962687349497502709413456509","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66640124635222298792902508374800337726","series":"1.2.826.0.1.3680043.8.498.24451015277571314447314357312644352242","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853","series":"1.2.826.0.1.3680043.8.498.16363006916399984278518005354558782662","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66841103065389483583108279900721522084","series":"1.2.826.0.1.3680043.8.498.10387224694921774954777339328400116961","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127","series":"1.2.826.0.1.3680043.8.498.11619637202488605590430668710697737794","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.66993076561387452114335652021593805727","series":"1.2.826.0.1.3680043.8.498.72625006746207267234868428065517380344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924","series":"1.2.826.0.1.3680043.8.498.86656816201474970522293043830946351857","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774","series":"1.2.826.0.1.3680043.8.498.11649991991434619892200264178265226078","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258","series":"1.2.826.0.1.3680043.8.498.89157056020900087172156914933668916321","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67374476726034608591738910363595695123","series":"1.2.826.0.1.3680043.8.498.77752410427617941380463794338297772545","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67397756161011684282778294948252031539","series":"1.2.826.0.1.3680043.8.498.22507961417250728785005567246678085153","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.54346729839030112360250896391534309983","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67498362978754615554767658151787188674","series":"1.2.826.0.1.3680043.8.498.46312128413889003333761558222902613054","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.73235436060469364028937479210954377254","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451","series":"1.2.826.0.1.3680043.8.498.29202133927078620283906032779898596402","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67690090498828405347380696553164435363","series":"1.2.826.0.1.3680043.8.498.53864293796081830276975033758827650128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67739091465243228820766982459725849814","series":"1.2.826.0.1.3680043.8.498.31299519534959113542930898261238802081","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848","series":"1.2.826.0.1.3680043.8.498.12216737799527645924118226111758152971","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67817378785531423675449308236680249899","series":"1.2.826.0.1.3680043.8.498.11744383812209394815133281031996428989","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67875980316053973061252844698884229168","series":"1.2.826.0.1.3680043.8.498.23031722909840698572229158615423899669","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.37236668619325387160420328376404405638","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879","series":"1.2.826.0.1.3680043.8.498.11201827604048943303444099811207690832","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67981539513814867568631217960473023750","series":"1.2.826.0.1.3680043.8.498.83130986690483914017925862088783851579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68098431505945078581285587720599762771","series":"1.2.826.0.1.3680043.8.498.17012437102052749790447173067534636682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147","series":"1.2.826.0.1.3680043.8.498.72454783953058519844740110840835845449","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.11631160978453599299342691317264410023","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179","series":"1.2.826.0.1.3680043.8.498.51045261746984269783924810704105846339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.31449246226241939401705364671546291899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68263925314265871578642679583159577645","series":"1.2.826.0.1.3680043.8.498.12241553563203853791005501041617283262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68304853223764870957854101604223809218","series":"1.2.826.0.1.3680043.8.498.10691364585849580554421330988389048401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68434116759606759586218589635360502038","series":"1.2.826.0.1.3680043.8.498.12190767273216890810722745303577048494","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68435927323644457286635872824563629385","series":"1.2.826.0.1.3680043.8.498.57395278214033364260511828049134372978","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68504220056564878621959394439671931747","series":"1.2.826.0.1.3680043.8.498.84770663397378591186771965882397035051","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462","series":"1.2.826.0.1.3680043.8.498.10032396632993060281858093554269204879","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68593524008570334427931338225473487643","series":"1.2.826.0.1.3680043.8.498.85503731083633113725703261114876089947","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68668620295929607459355154952035421306","series":"1.2.826.0.1.3680043.8.498.28332041300272622043239168362653291447","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68744926735490285461495285547312308522","series":"1.2.826.0.1.3680043.8.498.86179520084711768719113574274327912864","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335","series":"1.2.826.0.1.3680043.8.498.76123865232882119139470063579377656630","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.56413157321973387929839918876385440071","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960","series":"1.2.826.0.1.3680043.8.498.19235121870397736279796190725857875523","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840","series":"1.2.826.0.1.3680043.8.498.94785646669929879730963464434830005266","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819","series":"1.2.826.0.1.3680043.8.498.88994166362456138288389295970487292273","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703","series":"1.2.826.0.1.3680043.8.498.77302951126851460495729794069452787593","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455","series":"1.2.826.0.1.3680043.8.498.12601349571192885112616803874110138652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69311065409015951283858403030798356475","series":"1.2.826.0.1.3680043.8.498.34812662377553744203858614646225670919","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69397705155385306818511499752200941968","series":"1.2.826.0.1.3680043.8.498.85528417357662351465916243568924561591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69429384137362531712760113705595820981","series":"1.2.826.0.1.3680043.8.498.66770562016850922743811782912835463873","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69514709065182531331987458626496800737","series":"1.2.826.0.1.3680043.8.498.28668754214310116003106926651297272584","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384","series":"1.2.826.0.1.3680043.8.498.91686343460974739197446999362698620410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69654898267139720912940475772303404772","series":"1.2.826.0.1.3680043.8.498.13304560153473432627374597991175525873","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69717507929047897681405659943474469018","series":"1.2.826.0.1.3680043.8.498.87232920599237877561900363672024515410","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69730556680449158538173708009262014812","series":"1.2.826.0.1.3680043.8.498.12172868593979401142662219675042291519","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494","series":"1.2.826.0.1.3680043.8.498.12619565303347867263478305304684733860","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69954533310505596128507234525019777805","series":"1.2.826.0.1.3680043.8.498.64742273126525403535067237319002240165","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70125113032990356507120856030706104508","series":"1.2.826.0.1.3680043.8.498.10455464907477204793529858653158147940","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70212141059019236877414201247791694101","series":"1.2.826.0.1.3680043.8.498.32564343371137038832769746390700352721","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063","series":"1.2.826.0.1.3680043.8.498.58942967152728417981440658684975475631","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260","series":"1.2.826.0.1.3680043.8.498.10177732173338489898526416475030616559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70427947434642957729711329062828233047","series":"1.2.826.0.1.3680043.8.498.29749771348123533151669824071032288263","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.10865332009625927915428143800717235186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314","series":"1.2.826.0.1.3680043.8.498.10698888545593206036282755273266809767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70609649448269244608860767754947143395","series":"1.2.826.0.1.3680043.8.498.12598646489189005219313677302082830592","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70691305452244378255646875206864340458","series":"1.2.826.0.1.3680043.8.498.13252942853855358781173260200485153666","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70829434408808897601630953382239533220","series":"1.2.826.0.1.3680043.8.498.33066897921600879717396114242606511088","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526","series":"1.2.826.0.1.3680043.8.498.12025167786811786178111467757424555896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","series":"1.2.826.0.1.3680043.8.498.66515782659441534392812075331697920648","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70913971933753517251340687745718346828","series":"1.2.826.0.1.3680043.8.498.10311693038864023859428477778277983540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315","series":"1.2.826.0.1.3680043.8.498.78833391589772788456322771447658736245","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261","series":"1.2.826.0.1.3680043.8.498.23750182733761052169753659094434191242","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71001339006150287453095055802545712130","series":"1.2.826.0.1.3680043.8.498.18222102490588996924619295207306047243","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71063550263564056105844586167361750567","series":"1.2.826.0.1.3680043.8.498.64743817541248850748777472230119829333","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71180975285730441100082825359032677931","series":"1.2.826.0.1.3680043.8.498.55327046381604476100739142659680675453","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71355457409240695394093233905149541215","series":"1.2.826.0.1.3680043.8.498.44492382391192929063732501491993087733","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.10625483737156461198922750615764930315","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.71622107620007706778976946784787365505","series":"1.2.826.0.1.3680043.8.498.42831976851723896252231188230257381738","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585","series":"1.2.826.0.1.3680043.8.498.77875355663473865047515221479951807964","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71688651674041466985305836912343513997","series":"1.2.826.0.1.3680043.8.498.81716121527307093957510649225340921524","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504","series":"1.2.826.0.1.3680043.8.498.12497199821283410350560656990717229657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437","series":"1.2.826.0.1.3680043.8.498.10917918545499235402618243917661516019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71753603955749189961638324806586359888","series":"1.2.826.0.1.3680043.8.498.34661016677190882207404030705552773127","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71814919885777053270239861047114794237","series":"1.2.826.0.1.3680043.8.498.22460454897530289483372375897303304447","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71836684300308056297405632410209248276","series":"1.2.826.0.1.3680043.8.498.12267524159641239784534977875663839967","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71898908031968158312013185809213669768","series":"1.2.826.0.1.3680043.8.498.32088876397023987488368984850544262077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627","series":"1.2.826.0.1.3680043.8.498.62507196110587224915433952269544774304","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.35679368128328302904752944261821888762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72095214441332094906401331319197122529","series":"1.2.826.0.1.3680043.8.498.95058965298993768473305425866050807021","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72172265826902387784016098389863005666","series":"1.2.826.0.1.3680043.8.498.47838263261178653738294054895422287121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774","series":"1.2.826.0.1.3680043.8.498.87026161474275179389102010309270639563","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72252601583562621620422058467969183753","series":"1.2.826.0.1.3680043.8.498.73508334366196945399948577692904789302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72270915206075693056344475438055463113","series":"1.2.826.0.1.3680043.8.498.82724435399595752600974638542840120131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72342425990274562150756501476989691572","series":"1.2.826.0.1.3680043.8.498.17918191068061474713431626913700573689","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.56735010118341192858939466354489441056","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72453426317225885428514595399496580896","series":"1.2.826.0.1.3680043.8.498.12735544341190179814573504097225815090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72495677804641720406765749568114476096","series":"1.2.826.0.1.3680043.8.498.37948106174338505961542370163193204988","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72576069358772864331810785663326683534","series":"1.2.826.0.1.3680043.8.498.26887391387983935913263559668054690624","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72653666963736486779382314125782779004","series":"1.2.826.0.1.3680043.8.498.11362854644150465317095255496758137250","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585","series":"1.2.826.0.1.3680043.8.498.11559042079639193000113154150057669425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309","series":"1.2.826.0.1.3680043.8.498.40903114975609029357805140860099570851","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230","series":"1.2.826.0.1.3680043.8.498.23252727397863615864130172724027481683","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72880034930990967339546757487847803284","series":"1.2.826.0.1.3680043.8.498.12727419215265919304381508028645127897","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492","series":"1.2.826.0.1.3680043.8.498.11389986159238054344919458971173784242","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72943191222961132416678221885831944435","series":"1.2.826.0.1.3680043.8.498.13040212818534732735439762770324014783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73033521237848260228708170280586043408","series":"1.2.826.0.1.3680043.8.498.13983384413883779931852304872071648789","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73070182506490199787361298167985971577","series":"1.2.826.0.1.3680043.8.498.70723594464298027885849633227957834661","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73083786204651635096412213233334934941","series":"1.2.826.0.1.3680043.8.498.39237558209156662885683981750472338396","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73194579460722181989151410198309642266","series":"1.2.826.0.1.3680043.8.498.10863876454137158625241855616513313689","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73205916973275504240156555900460720571","series":"1.2.826.0.1.3680043.8.498.76877992430641113479437288060775923675","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73241519025825862373621891294281253158","series":"1.2.826.0.1.3680043.8.498.80652427712183119050817258688449519294","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751","series":"1.2.826.0.1.3680043.8.498.12451453468534525403298376943605595893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074","series":"1.2.826.0.1.3680043.8.498.33935933612343911675990820466278549021","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73394992092536626285935384324191560182","series":"1.2.826.0.1.3680043.8.498.12414531358843313094822818001807225300","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73477249559479870799427355988138990349","series":"1.2.826.0.1.3680043.8.498.38661605358386736441022666610459077178","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615","series":"1.2.826.0.1.3680043.8.498.77493489505412789171430216419271177616","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220","series":"1.2.826.0.1.3680043.8.498.10348071327293290966523579991979381444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73567807253694728043552196136704821457","series":"1.2.826.0.1.3680043.8.498.41252421516459328577811663671803574560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115","series":"1.2.826.0.1.3680043.8.498.34425117703828161069141876665526630766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73792361507751648386401969571717557389","series":"1.2.826.0.1.3680043.8.498.10063731081890047751464503910956626637","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73869775636181013203432845292462040922","series":"1.2.826.0.1.3680043.8.498.10920769936631817743451008600808472722","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73983081622010008601849687650985260980","series":"1.2.826.0.1.3680043.8.498.11154694393672244220388674625214533322","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74052473719853343027548507306186291134","series":"1.2.826.0.1.3680043.8.498.13094096533304846113379485116266339190","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74095402538454760179339842746572802563","series":"1.2.826.0.1.3680043.8.498.22537317190739368825598872304676377224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74160604730133050784449689952695940558","series":"1.2.826.0.1.3680043.8.498.65998044759212234190171057721137980305","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74206939384612669977629625310277500302","series":"1.2.826.0.1.3680043.8.498.32788100201788624117375266718008261274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74222284584657051737421776919599206132","series":"1.2.826.0.1.3680043.8.498.94041369938572178894062054104998115901","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74243110959141459268838217468465880820","series":"1.2.826.0.1.3680043.8.498.26977274722728899649111534053702796957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465","series":"1.2.826.0.1.3680043.8.498.50836390245681571445308412021444522012","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74332475620045145049628676751586693385","series":"1.2.826.0.1.3680043.8.498.31947675238495457665511911486744885636","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326","series":"1.2.826.0.1.3680043.8.498.66771018895045760672065118431087325546","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661","series":"1.2.826.0.1.3680043.8.498.10774027666193881155759398415390218393","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74622031396885197580453304563088863451","series":"1.2.826.0.1.3680043.8.498.11700076446771170224966977910695342231","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74688733778107958348746767768064374883","series":"1.2.826.0.1.3680043.8.498.30985189160377439852460148369056564679","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","series":"1.2.826.0.1.3680043.8.498.80426876457332459383899034047749222360","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738","series":"1.2.826.0.1.3680043.8.498.50523458396466286300281783993797071182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74939269621045823332837094516056211606","series":"1.2.826.0.1.3680043.8.498.95031228248720229011974241826034864485","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74952769952682175467667812958859010329","series":"1.2.826.0.1.3680043.8.498.49625673330324116561959998712970089773","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716","series":"1.2.826.0.1.3680043.8.498.44578504073510854179436837334244323888","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190","series":"1.2.826.0.1.3680043.8.498.11625457303004368642031400585358126215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638","series":"1.2.826.0.1.3680043.8.498.19055497418111707641341587565471612633","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691","series":"1.2.826.0.1.3680043.8.498.41880720647128233523840466617565611270","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75420186662442688955616630735303321846","series":"1.2.826.0.1.3680043.8.498.49118395816778787024167077784925564044","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.21450191861413509861211568935024698089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798","series":"1.2.826.0.1.3680043.8.498.11718024462343690891574900055143140803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75577846246542841539613557336025271434","series":"1.2.826.0.1.3680043.8.498.33397285285681571410107359215679000860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75627574440860942023321071382766359827","series":"1.2.826.0.1.3680043.8.498.58224143122402950059678672786789650732","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75688484058613633923943882730405145535","series":"1.2.826.0.1.3680043.8.498.11992547544815615461298179015377870061","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75742195884147990442829552992649204078","series":"1.2.826.0.1.3680043.8.498.13632064901744352402565351678529296399","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75760698509105388883811155432691667447","series":"1.2.826.0.1.3680043.8.498.36503266320810838690806080156047457894","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75800418125597325281807386716279941068","series":"1.2.826.0.1.3680043.8.498.22883521491633323484458170149062317420","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75926011938952404257302292495731660728","series":"1.2.826.0.1.3680043.8.498.57564823557183240281629499154643339659","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76012896432318288623173593280895772876","series":"1.2.826.0.1.3680043.8.498.73215409962997775568722179437908633030","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088","series":"1.2.826.0.1.3680043.8.498.13327539916047437302761353819957215036","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.91997321929493269866593913584164113588","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76167153566894354739332364376483514559","series":"1.2.826.0.1.3680043.8.498.12261153692903120965071188309170886659","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76242245340157474968574926079920787199","series":"1.2.826.0.1.3680043.8.498.68904965436423298027902675671313785108","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76344826994380951498651448092980303697","series":"1.2.826.0.1.3680043.8.498.41366565336496253841847902256289848873","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76514504104121634569792846980451896030","series":"1.2.826.0.1.3680043.8.498.83459826600516305978207961132246739691","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120","series":"1.2.826.0.1.3680043.8.498.10426516668056198112366031072480143102","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636","series":"1.2.826.0.1.3680043.8.498.10091920228201404584030104082141315492","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323","series":"1.2.826.0.1.3680043.8.498.12430027056839893949837688130167884582","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789","series":"1.2.826.0.1.3680043.8.498.54197545302277573007080266298433000224","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.76862911430418639277808760660745763387","series":"1.2.826.0.1.3680043.8.498.53106916934237313778776656083513380618","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939","series":"1.2.826.0.1.3680043.8.498.17833224052981955678447450995853530899","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76993922254007307143291254174266755701","series":"1.2.826.0.1.3680043.8.498.69821789247282628496611675579208063469","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.24823180421157623253115954570018126431","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536","series":"1.2.826.0.1.3680043.8.498.40840809953611428085023070474190465458","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855","series":"1.2.826.0.1.3680043.8.498.81632725945746211863859873277852110793","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.11342105499090312779164985459635198058","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77261014467268552758826551926128217025","series":"1.2.826.0.1.3680043.8.498.23127779601923449366206637244676359990","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.32508264269837329551097887281787091281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136","series":"1.2.826.0.1.3680043.8.498.53879049114650315823127980262689458854","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839","series":"1.2.826.0.1.3680043.8.498.18614121448197327554448548256747856230","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77370119801633353780677378448761665685","series":"1.2.826.0.1.3680043.8.498.33094918834447258632631606778253064475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77391801789875882109036004511558148658","series":"1.2.826.0.1.3680043.8.498.89115047894747930772708315835100271372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.69506505647776954385374600297464413231","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971","series":"1.2.826.0.1.3680043.8.498.66213856727957564772704255694607426308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320","series":"1.2.826.0.1.3680043.8.498.18634026080600672477552122784204966188","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713","series":"1.2.826.0.1.3680043.8.498.15068881868588002852865241814610982562","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146","series":"1.2.826.0.1.3680043.8.498.46547966283908465505854784634981320162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77626578713099735143745396709359607445","series":"1.2.826.0.1.3680043.8.498.60143347930555299752247887975033513736","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77716260980321491689913730053309892120","series":"1.2.826.0.1.3680043.8.498.55873435841112104978829390412266329746","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77753753872601674668756777373761660873","series":"1.2.826.0.1.3680043.8.498.11301685336080953376509967173297504736","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77800740811302686332282410929816056283","series":"1.2.826.0.1.3680043.8.498.57049704599729575276490901751360243742","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398","series":"1.2.826.0.1.3680043.8.498.98969943254028277789214436154499594520","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77893704208186903913532950495036889026","series":"1.2.826.0.1.3680043.8.498.25375977763263963134865159514140165597","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.11202952790868103383350951342340861694","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.13008015395074886554658244032093917109","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78083057157844664959843364584162587026","series":"1.2.826.0.1.3680043.8.498.29085682831869619748559824196059220192","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456","series":"1.2.826.0.1.3680043.8.498.80694240921235522334851360396284760777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78236036694398253902871510456985685293","series":"1.2.826.0.1.3680043.8.498.77963622743321207438825183345748936219","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586","series":"1.2.826.0.1.3680043.8.498.37703009186488753714438880349747141714","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268","series":"1.2.826.0.1.3680043.8.498.19136650837090895214821749451270999941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78475145549489837635803968565380247517","series":"1.2.826.0.1.3680043.8.498.87432444735559104188888846872469365756","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828","series":"1.2.826.0.1.3680043.8.498.51594414677993339088394514174201163950","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78646473961447523033955716877524794707","series":"1.2.826.0.1.3680043.8.498.53410227961277878410329293915577792619","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78667999747216717251638754803697959701","series":"1.2.826.0.1.3680043.8.498.87708548160868029129363084789815037970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78757011872917563830995996191850541453","series":"1.2.826.0.1.3680043.8.498.15213849669243167363561627739687587875","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78800983923143929298359366572020370031","series":"1.2.826.0.1.3680043.8.498.11618086813746276690171702849313901601","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231","series":"1.2.826.0.1.3680043.8.498.95082199162561012718590100997093835515","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79021499290691125513421477204330532755","series":"1.2.826.0.1.3680043.8.498.54425545254595855977561202768427487509","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79077219037094025366585138112899760966","series":"1.2.826.0.1.3680043.8.498.10985772276689253392309698046726542896","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79117047831806895544174734229992359986","series":"1.2.826.0.1.3680043.8.498.25279317337825031775417583603441147073","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.72829817500734506111899867477962910056","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317","series":"1.2.826.0.1.3680043.8.498.11619806721910516093822150765740881128","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79247871688929479411597859576284194165","series":"1.2.826.0.1.3680043.8.498.85703980753816571484340720998427564896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79274948549030940175536603063980168192","series":"1.2.826.0.1.3680043.8.498.84752435707155641590734898355592177881","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79292862882980756453890778996691993351","series":"1.2.826.0.1.3680043.8.498.53636994615213265940594482789781241739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079","series":"1.2.826.0.1.3680043.8.498.14221923483037933809847772490637262961","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79551456190098389081551755482042135791","series":"1.2.826.0.1.3680043.8.498.11389092962714013667261023513000962456","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79582457162604674356422857305405255763","series":"1.2.826.0.1.3680043.8.498.93234114535444031537995336072612042920","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79636443202100988025435536780567446582","series":"1.2.826.0.1.3680043.8.498.12476716683737332319462982244374557979","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688","series":"1.2.826.0.1.3680043.8.498.15454163485933290186202657924300566922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79798779812052605703861699094268016306","series":"1.2.826.0.1.3680043.8.498.64410132725489840253677263673675921323","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79991838737793096019496013540529642900","series":"1.2.826.0.1.3680043.8.498.10756670816928145777623613213754822855","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80100785345864355815698147663964005920","series":"1.2.826.0.1.3680043.8.498.65764708555178843024129879588170989798","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128","series":"1.2.826.0.1.3680043.8.498.47798047231427994744312211340332482960","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","series":"1.2.826.0.1.3680043.8.498.23549877034569314877573478288071642740","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80275294885165896124489476449616436102","series":"1.2.826.0.1.3680043.8.498.12843755883604470387897670658967013015","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80292580287849976694993628109721422478","series":"1.2.826.0.1.3680043.8.498.39772007426652012413324414561964299669","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146","series":"1.2.826.0.1.3680043.8.498.18460713280077733900668632748561237200","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80391378873630377472192017862302619947","series":"1.2.826.0.1.3680043.8.498.18979249056911895380221252979341779314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.12469889792738163627740372409201276089","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286","series":"1.2.826.0.1.3680043.8.498.47124723659005253287451500260336283829","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80589796918823112368758999678369292754","series":"1.2.826.0.1.3680043.8.498.71771781562165030728502111848997741421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307","series":"1.2.826.0.1.3680043.8.498.36437528437662069991535728620916903311","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925","series":"1.2.826.0.1.3680043.8.498.29013428747777778112006935210105261345","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80671597537348112977780954910198682272","series":"1.2.826.0.1.3680043.8.498.87472148924091904893454307605583707039","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80735285867775404153100236162962472952","series":"1.2.826.0.1.3680043.8.498.74477873758959689716759440910516029199","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704","series":"1.2.826.0.1.3680043.8.498.48418634714338866127373932657817838084","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80861108320341696163987804408524915999","series":"1.2.826.0.1.3680043.8.498.78628782566688474740162205148817563835","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834","series":"1.2.826.0.1.3680043.8.498.93475193632138921088843198476353100587","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81012548926053584036386949658594985125","series":"1.2.826.0.1.3680043.8.498.16285125730855516900471559718517257125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81031057654953109295788159826841315360","series":"1.2.826.0.1.3680043.8.498.65527086195929520106500328697488187615","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81103809943451790668501089584752866199","series":"1.2.826.0.1.3680043.8.498.55156729103258607655507426510295084154","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81248820934296461385324385812581970340","series":"1.2.826.0.1.3680043.8.498.12204093881583172544548890521278117112","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81327951282145199298572980264557740321","series":"1.2.826.0.1.3680043.8.498.36095167077783345539157835743962771461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81347841116019279082226143308588516787","series":"1.2.826.0.1.3680043.8.498.10264273651141823496127762043131445004","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669","series":"1.2.826.0.1.3680043.8.498.21807506332695750363757013415147545715","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81466575994184158172641823163654418018","series":"1.2.826.0.1.3680043.8.498.11476030943211531012536736357678858012","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563","series":"1.2.826.0.1.3680043.8.498.43064716921017676999782827926594756375","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166","series":"1.2.826.0.1.3680043.8.498.56865376584453129369250882021340154021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81651132686162360431299931833372141617","series":"1.2.826.0.1.3680043.8.498.49785186593642092527116972318536512809","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.49294732499116516032345128989257882402","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81738571920712657790861120957086010772","series":"1.2.826.0.1.3680043.8.498.12275778125606185076128074386672339659","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81793454171031578492222004629265937656","series":"1.2.826.0.1.3680043.8.498.12195466511883083134006636642461680982","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81797437102217585216278133168092577952","series":"1.2.826.0.1.3680043.8.498.73301420630287456703160224414420919389","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236","series":"1.2.826.0.1.3680043.8.498.10633369153197123770487217941923058907","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81927409938015158663782061022054347955","series":"1.2.826.0.1.3680043.8.498.75246292585761236295055436966284015992","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277","series":"1.2.826.0.1.3680043.8.498.41242287001126099617212379571727168903","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894","series":"1.2.826.0.1.3680043.8.498.94626979394672032998905435628653366748","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82053853036886620816821481139413594407","series":"1.2.826.0.1.3680043.8.498.13009462421912372861600089610124267368","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426","series":"1.2.826.0.1.3680043.8.498.10635017995303866624489660329043588758","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.30205011752939967279543537379010794909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181","series":"1.2.826.0.1.3680043.8.498.10568996911457291723691785963888054957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82440268452598207215458064865523988222","series":"1.2.826.0.1.3680043.8.498.97884492850738775569493591351774763471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874","series":"1.2.826.0.1.3680043.8.498.37045372354437856593828008458745981552","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82548251499640077572026404183402934330","series":"1.2.826.0.1.3680043.8.498.11990348412552130864400517394306281424","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82582558624896662901163827145277490232","series":"1.2.826.0.1.3680043.8.498.14081516246776542894770066494703301017","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714","series":"1.2.826.0.1.3680043.8.498.11228572219073597760950643531724422320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538","series":"1.2.826.0.1.3680043.8.498.51868683876515563231455370893755876675","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400","series":"1.2.826.0.1.3680043.8.498.25890157992535818917052715866009181369","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82967086493280898295669930903240677729","series":"1.2.826.0.1.3680043.8.498.98270105135860045580993256416990691521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83028939839767027342482365275441078777","series":"1.2.826.0.1.3680043.8.498.79356857167205549309691631478131407949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783","series":"1.2.826.0.1.3680043.8.498.50372128740131407441319949403980310098","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83108474538388106770920972885900596508","series":"1.2.826.0.1.3680043.8.498.36093716085054569742306678961307831596","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83220896521302064187926748035734200732","series":"1.2.826.0.1.3680043.8.498.31607271877684371913918498272680354960","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83237706274929081175032033288272528919","series":"1.2.826.0.1.3680043.8.498.28883146640527867006896372321038842788","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83271280040777227673434567660749430744","series":"1.2.826.0.1.3680043.8.498.93685361276163851700888519241647140609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83456649088980869429062498344457162631","series":"1.2.826.0.1.3680043.8.498.11868139917915383063349758092604065686","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986","series":"1.2.826.0.1.3680043.8.498.84006227723803319680019793674085294608","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.11811253223921678890380648500686192415","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83747418172021065605877274971789603771","series":"1.2.826.0.1.3680043.8.498.33699222672574848167977614612582297181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83956129905762530953192730040209719910","series":"1.2.826.0.1.3680043.8.498.50939810170493514425214557938011033201","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783","series":"1.2.826.0.1.3680043.8.498.47721780321107259642394898895088077639","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220","series":"1.2.826.0.1.3680043.8.498.92037252229392206999547571997540029415","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488","series":"1.2.826.0.1.3680043.8.498.84039324418244493559746559176056499531","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724","series":"1.2.826.0.1.3680043.8.498.62604710473611234068051600368989464805","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259","series":"1.2.826.0.1.3680043.8.498.10827263939960769727435881137721410996","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84483080389344581106129884869678629612","series":"1.2.826.0.1.3680043.8.498.28334461853763977328452819111358915986","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84630493975910180537831353884518764364","series":"1.2.826.0.1.3680043.8.498.36365750546880568795579534871319206157","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195","series":"1.2.826.0.1.3680043.8.498.85323147993447425048072926578590793295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.11086545566910011247716170962453698871","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518","series":"1.2.826.0.1.3680043.8.498.40880644513173049911520157577567668688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.11570014931048528586446059636628173885","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548","series":"1.2.826.0.1.3680043.8.498.10167868607217458122078007730614010599","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85095389649021071892478435298763928925","series":"1.2.826.0.1.3680043.8.498.84248746768262100469301498515197645662","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85137704768728261642637391657270006448","series":"1.2.826.0.1.3680043.8.498.12241796296299270377961475643224951514","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","series":"1.2.826.0.1.3680043.8.498.45477239265096136551433173907103114518","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85333821516157983299252782678788406019","series":"1.2.826.0.1.3680043.8.498.14434786799957322425595376932205260258","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85391052076760686301971262244561025345","series":"1.2.826.0.1.3680043.8.498.13164978557358821214210482010699116388","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85416107585703628549041626609272707431","series":"1.2.826.0.1.3680043.8.498.83220999489977215221294519339762508279","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327","series":"1.2.826.0.1.3680043.8.498.27664922959458763149095704328534509822","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85499036441349866810703945289793057098","series":"1.2.826.0.1.3680043.8.498.19246259420842247772845127360458141612","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85602291032341007550376696695283794665","series":"1.2.826.0.1.3680043.8.498.24875781295944541910955410875348458398","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533","series":"1.2.826.0.1.3680043.8.498.86829916776652768525923325437434421076","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.10403404917678967163632018085748325016","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221","series":"1.2.826.0.1.3680043.8.498.81974047601967057282078566700302812728","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85735729179216019275685564112535593988","series":"1.2.826.0.1.3680043.8.498.21673779124658422734387885328625737805","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501","series":"1.2.826.0.1.3680043.8.498.15896158049991245133301480994001364406","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85879572979834912642178551098379145607","series":"1.2.826.0.1.3680043.8.498.20988111362255799033301478614091703140","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290","series":"1.2.826.0.1.3680043.8.498.10309354108159093407911675069275843467","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85943737587102032370951456635167782343","series":"1.2.826.0.1.3680043.8.498.10335555054928741153620086817417866255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85992037603186775437749979929724514304","series":"1.2.826.0.1.3680043.8.498.81382073306324455376965698233499417998","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091","series":"1.2.826.0.1.3680043.8.498.12229428925120150174124839641901129610","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86067849233330188988240710585112183308","series":"1.2.826.0.1.3680043.8.498.10796697725572674302483805112481424521","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915","series":"1.2.826.0.1.3680043.8.498.67247379040283313063021945042121981131","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86239793008693746155478763153154040128","series":"1.2.826.0.1.3680043.8.498.89676638518543935801069871010178592571","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86273853115722509980053190345047859057","series":"1.2.826.0.1.3680043.8.498.14127686382091630402276071386135750092","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86285715293767810771609523819042157072","series":"1.2.826.0.1.3680043.8.498.40211573928778209708977542011769454746","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601","series":"1.2.826.0.1.3680043.8.498.55717154019717077737583062562413059503","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86405969333062807387894608315897106350","series":"1.2.826.0.1.3680043.8.498.14621844443382834301627397221567980607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86416984885007871128151024515780322767","series":"1.2.826.0.1.3680043.8.498.12501363222599823505936923664284435705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86435890312227285216810145658548164773","series":"1.2.826.0.1.3680043.8.498.11911678759954903566988332591493979302","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133","series":"1.2.826.0.1.3680043.8.498.92211960487474598848166482429925478035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86625876521226465945114246972542881955","series":"1.2.826.0.1.3680043.8.498.39147383008511362457857080353767559649","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.12183977682610016889339289574042190337","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","series":"1.2.826.0.1.3680043.8.498.42451997021748211255954641885421797696","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409","series":"1.2.826.0.1.3680043.8.498.11504035152145730194393793003361262158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937","series":"1.2.826.0.1.3680043.8.498.11147486778761255883207139795231394027","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87101276883427329522365304528449003589","series":"1.2.826.0.1.3680043.8.498.94814108642837836489523030666201682817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87198191797585172912979594638355685739","series":"1.2.826.0.1.3680043.8.498.96131416298582963128783630031245606980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87214831397290763080243320084625847148","series":"1.2.826.0.1.3680043.8.498.20141496751615304080047044578660894691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87241650441242408799533998101943024283","series":"1.2.826.0.1.3680043.8.498.27505202292422415943021967237870191840","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87262205711949358943515326577886508585","series":"1.2.826.0.1.3680043.8.498.72810320791793032751175801806804561287","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.88919541785385002414903531908299271460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155","series":"1.2.826.0.1.3680043.8.498.14534693076291100409998862601428211851","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757","series":"1.2.826.0.1.3680043.8.498.10519364878297196702912849269962262764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.68814596151353548864623642754281200813","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110","series":"1.2.826.0.1.3680043.8.498.86743044604912670055203470648536531262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348","series":"1.2.826.0.1.3680043.8.498.11211913216545394450725197091282862155","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87761573479019057997334508266888503487","series":"1.2.826.0.1.3680043.8.498.11593657581716220244445985417722685883","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87799106106199445436621174506197684035","series":"1.2.826.0.1.3680043.8.498.12097470461878764565836463479658566827","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87876800169766592603600598167358214111","series":"1.2.826.0.1.3680043.8.498.13220997570249376692335060590171088070","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523","series":"1.2.826.0.1.3680043.8.498.13369137240875167732318185714618938255","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88070896604159829163161019117970440189","series":"1.2.826.0.1.3680043.8.498.10421802825459555793784795464011645540","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88114918358665488269761388157480523350","series":"1.2.826.0.1.3680043.8.498.85106349993599790514035796261571981454","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685","series":"1.2.826.0.1.3680043.8.498.86424377002945448890561156480430598873","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88439732365857773633852840383740478214","series":"1.2.826.0.1.3680043.8.498.50615845862697221289599273537366924747","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.49331402471599556512629771187035191770","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88523737187772411911621088333701306485","series":"1.2.826.0.1.3680043.8.498.13126741595408225825152945368477214128","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560","series":"1.2.826.0.1.3680043.8.498.39546679642182632103839726109775503699","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88609906210780568637209291182572648571","series":"1.2.826.0.1.3680043.8.498.43179406803085529058831531019275803292","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88653085234639096911379842849193874077","series":"1.2.826.0.1.3680043.8.498.58224234075116053048907690289786633445","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88682169866098509464628948503453852928","series":"1.2.826.0.1.3680043.8.498.10424197981919376197223089597436220301","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015","series":"1.2.826.0.1.3680043.8.498.30401069781573447338735050481517299208","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.79840446333263648104839621457831341695","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.88827276474694165071267273766912871459","series":"1.2.826.0.1.3680043.8.498.66427415874153904376899117809231959725","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88874647535130028956725641820977918615","series":"1.2.826.0.1.3680043.8.498.61144959244960997441714914838822234564","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88899519473612920868099376939780201067","series":"1.2.826.0.1.3680043.8.498.14490307627670409361391309497207554747","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.58247834975996642632914258227157253428","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.81004829384088362603071917065957316030","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.69697840629822541536995998994826813807","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89190511197564127079697151213577962649","series":"1.2.826.0.1.3680043.8.498.12967022916510608343381445176013426443","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89247482662611898098489676481780144186","series":"1.2.826.0.1.3680043.8.498.10835560714447261138202554940121442263","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89307235192406983202868051697765144920","series":"1.2.826.0.1.3680043.8.498.58965874257917781388645379763464668043","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946","series":"1.2.826.0.1.3680043.8.498.10194741669017655263751732281471753643","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030","series":"1.2.826.0.1.3680043.8.498.30133596071525793612266168031258338764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.11283085428988504719256823665855223285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89569762088162031469299314053759758927","series":"1.2.826.0.1.3680043.8.498.84086868462093268047660581515730549339","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89584627944947813918297925813120247226","series":"1.2.826.0.1.3680043.8.498.31422620681884137380200185172998156517","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","series":"1.2.826.0.1.3680043.8.498.13577859742968563226613957126735256631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89829843399218627892046362324057240535","series":"1.2.826.0.1.3680043.8.498.23065705253444354430828217544757708922","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.12777917608699964730170252436606963074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89953731420879076053772747816096629231","series":"1.2.826.0.1.3680043.8.498.80695500189054006660776291319735338576","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280","series":"1.2.826.0.1.3680043.8.498.11509530729408783588665690810682298178","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90057130466053377217652879986591534745","series":"1.2.826.0.1.3680043.8.498.12600446116142828596509105791894997369","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90170179734833731758923964833173536172","series":"1.2.826.0.1.3680043.8.498.11533115230720143144337268246094418322","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90193654806390401129887447441469319421","series":"1.2.826.0.1.3680043.8.498.46415444395779464864169049464237681130","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704","series":"1.2.826.0.1.3680043.8.498.11836029054857716982743729017913547707","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90264300606047579088362539636050224478","series":"1.2.826.0.1.3680043.8.498.67745666979610364851537354592320926623","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767","series":"1.2.826.0.1.3680043.8.498.84482758825468431885247408395318186921","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90299615112155255774026824400461236845","series":"1.2.826.0.1.3680043.8.498.28428685234464628259777886212827217421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164","series":"1.2.826.0.1.3680043.8.498.74021936627911165906674164883736387402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90446842983860370183814559926724769347","series":"1.2.826.0.1.3680043.8.498.12384833700428344830039295002161119348","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272","series":"1.2.826.0.1.3680043.8.498.28025583952216818499436381048030807046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.36697052089950898349082816840894648402","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781","series":"1.2.826.0.1.3680043.8.498.33238712348389620451226307711900128613","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.46585491643085331988722703348452483500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90751504670572401790063555695835439317","series":"1.2.826.0.1.3680043.8.498.23580036786525362145347010286747656939","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90771070773751767226565553195436971014","series":"1.2.826.0.1.3680043.8.498.17363729593853874068410721418051877910","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506","series":"1.2.826.0.1.3680043.8.498.66509580531621408840798779175878218369","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91064647744940506223536204418057554573","series":"1.2.826.0.1.3680043.8.498.26855959362664793659179017399105829254","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698","series":"1.2.826.0.1.3680043.8.498.73980058175004290592489229078790077254","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089","series":"1.2.826.0.1.3680043.8.498.77793974189182729828658745910670094895","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645","series":"1.2.826.0.1.3680043.8.498.93999576120450815553804137726345495467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811","series":"1.2.826.0.1.3680043.8.498.66120939706626411143551710083021933853","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91323765268136520804399328350809179765","series":"1.2.826.0.1.3680043.8.498.11776355073687736917645830302215403564","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394","series":"1.2.826.0.1.3680043.8.498.47711543544189101715415199430394529049","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984","series":"1.2.826.0.1.3680043.8.498.45772227470305557091440633513938899082","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785","series":"1.2.826.0.1.3680043.8.498.85061789002872432184671387069716645966","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999","series":"1.2.826.0.1.3680043.8.498.55839928712513397208478049545615030341","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91606395836371290680039618164703651751","series":"1.2.826.0.1.3680043.8.498.75190010642084013272356400851777367107","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91660278231317256353374962767408532443","series":"1.2.826.0.1.3680043.8.498.12734201828126354649570141613860909097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91742054227008690570256869200116133922","series":"1.2.826.0.1.3680043.8.498.93851717829876032875761313694505707070","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593","series":"1.2.826.0.1.3680043.8.498.12678921811109425043739156989108348746","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91897419008563027309752122760214275182","series":"1.2.826.0.1.3680043.8.498.38969735588274625319444434328761979654","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395","series":"1.2.826.0.1.3680043.8.498.65976872668193408890927007356450346848","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91991985280680856830060431668414014465","series":"1.2.826.0.1.3680043.8.498.89946991196444802849938805122990022129","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92071199603599654827155753204585082303","series":"1.2.826.0.1.3680043.8.498.35841406816793769153952500834913848829","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025","series":"1.2.826.0.1.3680043.8.498.85123538825980697283443119345161511734","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671","series":"1.2.826.0.1.3680043.8.498.16264021251152327335963176324105610407","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92326467051364762529119362944559591665","series":"1.2.826.0.1.3680043.8.498.11386101335059306619663108181395671386","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.60451363959320663888178855957562676105","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92449204325697143336618138322280397765","series":"1.2.826.0.1.3680043.8.498.12732470285056741964302215573012109480","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92597065276819676534178545455668171085","series":"1.2.826.0.1.3680043.8.498.12297599839883695914288403559451370800","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92621720172216795135762712303124932801","series":"1.2.826.0.1.3680043.8.498.10601904997268302114806430517578049330","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330","series":"1.2.826.0.1.3680043.8.498.93636337750878891177315081030978884616","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433","series":"1.2.826.0.1.3680043.8.498.97071267990965754415204354781516664088","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401","series":"1.2.826.0.1.3680043.8.498.62944100833309673957375051884816389057","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92902692647544828045768164058889405369","series":"1.2.826.0.1.3680043.8.498.51218362607018277207654677083326044383","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305","series":"1.2.826.0.1.3680043.8.498.76370408169428562661092487428972211498","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93096918345463865518663434763610016449","series":"1.2.826.0.1.3680043.8.498.26726053640714468900813376708597611324","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945","series":"1.2.826.0.1.3680043.8.498.23724364813817826931351175726707187890","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053","series":"1.2.826.0.1.3680043.8.498.10824226251314333299736755919432419690","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032","series":"1.2.826.0.1.3680043.8.498.13257901435738622165826188615373193659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706","series":"1.2.826.0.1.3680043.8.498.86483485737768384302875579566641677756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93385433312281352893457225415812880948","series":"1.2.826.0.1.3680043.8.498.37650833662350394053588093836738402722","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93623421864452751381778712192879121271","series":"1.2.826.0.1.3680043.8.498.42033979413155554698119716772598967904","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93655619137163635289231840294252859840","series":"1.2.826.0.1.3680043.8.498.24024923226226294642373075936253282669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93722352425168280299901366261374293722","series":"1.2.826.0.1.3680043.8.498.34483235374201518056535409093302108219","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680","series":"1.2.826.0.1.3680043.8.498.10752606275413987927697937734894583564","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93921993164883084178408030385007465064","series":"1.2.826.0.1.3680043.8.498.46697612136010121030932017497298956585","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838","series":"1.2.826.0.1.3680043.8.498.12164498489874517556406576130970466666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303","series":"1.2.826.0.1.3680043.8.498.12941934310659148721298156261745030373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693","series":"1.2.826.0.1.3680043.8.498.13000971437803044171437101949736546955","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94255124739558547668304022649099251744","series":"1.2.826.0.1.3680043.8.498.12200166733046206891510258921555551757","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94296943940827583101208235208463322140","series":"1.2.826.0.1.3680043.8.498.90884457443016070007373451697144487628","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94381282809153270156987123429739805206","series":"1.2.826.0.1.3680043.8.498.12949461294539970593270787277246399951","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312","series":"1.2.826.0.1.3680043.8.498.97920511515278513771772493032450247932","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.13087931369504311832045711465110274649","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94601583997736688762302974573903982304","series":"1.2.826.0.1.3680043.8.498.47492616584849347954539801226537056865","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516","series":"1.2.826.0.1.3680043.8.498.55757236721518352793999649063734555158","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618","series":"1.2.826.0.1.3680043.8.498.14942837234494719657442423206133403113","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175","series":"1.2.826.0.1.3680043.8.498.93901156442969248292717514004654456994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94761755148413531477206243804832613511","series":"1.2.826.0.1.3680043.8.498.12936799633366834090515726459200550196","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472","series":"1.2.826.0.1.3680043.8.498.65050546655149399589684997082097647811","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.11279409496102876322262551363692002179","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94954489705497211843190734684710878341","series":"1.2.826.0.1.3680043.8.498.32333086551619757435278507389561127511","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95030266788619257057351879256576338162","series":"1.2.826.0.1.3680043.8.498.88037040248275699241316525810447842193","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95080449640993916851075185393444497308","series":"1.2.826.0.1.3680043.8.498.39629872246087647603597695273598428479","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982","series":"1.2.826.0.1.3680043.8.498.82170306501491432359492283202183074007","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","series":"1.2.826.0.1.3680043.8.498.84647892678460271030996023909134001419","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95256518245659249144569318023469597946","series":"1.2.826.0.1.3680043.8.498.11387127798614283781908639970771809919","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95324318295506605557933083155792587297","series":"1.2.826.0.1.3680043.8.498.12587263832874489504127314424284960586","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963","series":"1.2.826.0.1.3680043.8.498.72362992423581492090681531001941399611","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815","series":"1.2.826.0.1.3680043.8.498.19993597982239635855205759350357158029","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948","series":"1.2.826.0.1.3680043.8.498.90938726231608468154436855103374615320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310","series":"1.2.826.0.1.3680043.8.498.49042130062228908634654526354490367799","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95586477246200285088746818821808017494","series":"1.2.826.0.1.3680043.8.498.51617471471794813101231254553898289994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95601371640669355936785637162110963091","series":"1.2.826.0.1.3680043.8.498.36375391844995609714269271712170363026","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95653228095828585867177997067051877351","series":"1.2.826.0.1.3680043.8.498.14021518457585552263642891347787071616","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95720534093121754146413060811354489247","series":"1.2.826.0.1.3680043.8.498.29654645624577907658408642036620596363","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450","series":"1.2.826.0.1.3680043.8.498.13254456221673831415420405383398798715","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95796009319232592137564145023096016872","series":"1.2.826.0.1.3680043.8.498.91535539374765371519019778653044879605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951","series":"1.2.826.0.1.3680043.8.498.91796186514479848944831361578888961845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466","series":"1.2.826.0.1.3680043.8.498.11453414272136434165435447742216914170","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601","series":"1.2.826.0.1.3680043.8.498.56753469150559675099965653226638037060","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.10615333233251747030601663920740833365","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.97114157090003883902853936736895964187","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428","series":"1.2.826.0.1.3680043.8.498.19461017110548607382135841834653270783","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674","series":"1.2.826.0.1.3680043.8.498.11054554137126759026580377557872555571","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96483513352751590579274577125356217169","series":"1.2.826.0.1.3680043.8.498.23704487503074026760356291542562750203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.64977160362416586421803894165166645689","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801","series":"1.2.826.0.1.3680043.8.498.10809732956225758251379394970320187904","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96621557839848412555023112236627663339","series":"1.2.826.0.1.3680043.8.498.66408360861504431125541939333234024526","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186","series":"1.2.826.0.1.3680043.8.498.75738321577460165994754105481540776103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96859317329616917022799512740280754327","series":"1.2.826.0.1.3680043.8.498.65542083359051612941576915063510097467","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.22223462134148429086434286618106946698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623","series":"1.2.826.0.1.3680043.8.498.10949365902671650269493276049228670871","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97042446513400958172996637321308032556","series":"1.2.826.0.1.3680043.8.498.65116032498496702201099188802055518974","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523","series":"1.2.826.0.1.3680043.8.498.38120645547388638052175289796502769428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97156518925820521351080132062400795929","series":"1.2.826.0.1.3680043.8.498.29458402001104183547883060753451552653","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97234348688223610934999923021953923903","series":"1.2.826.0.1.3680043.8.498.43214986040350746563689131302991398555","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97259766233250760343109933426776941754","series":"1.2.826.0.1.3680043.8.498.98468686598996674524849524337928915087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662","series":"1.2.826.0.1.3680043.8.498.17378665248295186489494676065815966976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97348112008590316355392924312704081684","series":"1.2.826.0.1.3680043.8.498.24386983475502535371875748199831349067","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97400307568415006244714024769069514497","series":"1.2.826.0.1.3680043.8.498.62703830555748841413100581257181788639","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373","series":"1.2.826.0.1.3680043.8.498.67216294187156200937742497328418553392","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97500632594529766269369977508352975364","series":"1.2.826.0.1.3680043.8.498.76340467848998951978471641511398937962","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480","series":"1.2.826.0.1.3680043.8.498.83347437479325698054147753636508207109","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028","series":"1.2.826.0.1.3680043.8.498.56117969371982676650206206878068973454","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091","series":"1.2.826.0.1.3680043.8.498.35387644207431109053936085316112446182","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614","series":"1.2.826.0.1.3680043.8.498.14241794714968084682863886357311642490","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97759980486481979519068526243106370997","series":"1.2.826.0.1.3680043.8.498.25376675413399962132143022793449759910","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97849030682292314880463675622456900933","series":"1.2.826.0.1.3680043.8.498.96365016888418252852552567787362402240","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.46066382287111309319960146442580223853","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98027050915885651900992366068887550187","series":"1.2.826.0.1.3680043.8.498.92280776906360075436663261318057932495","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381","series":"1.2.826.0.1.3680043.8.498.13168053550743659890377541600167899786","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354","series":"1.2.826.0.1.3680043.8.498.26963320992729255898694047768948557777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589","series":"1.2.826.0.1.3680043.8.498.17794462715156055975994646758574924012","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98146155044178593810385943365143374814","series":"1.2.826.0.1.3680043.8.498.56568262283933983237645788179645690672","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938","series":"1.2.826.0.1.3680043.8.498.62661560094602603779469299046906497961","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98315778542448798493600485116505234132","series":"1.2.826.0.1.3680043.8.498.19036925019599855767946820948664672173","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98343185597651455245996667389303915779","series":"1.2.826.0.1.3680043.8.498.10649946530349420302652320954747006492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225","series":"1.2.826.0.1.3680043.8.498.49201436250707533811257006610044467277","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764","series":"1.2.826.0.1.3680043.8.498.61323586466521802116408482256596179062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98752901295449528283322564256444025625","series":"1.2.826.0.1.3680043.8.498.98663216110134177214125182157425888021","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186","series":"1.2.826.0.1.3680043.8.498.48151331573804283241992300119637586805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98828753472895685940998888332947119084","series":"1.2.826.0.1.3680043.8.498.49215352028913324029897382774574696895","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98890851347757949328159691828768030366","series":"1.2.826.0.1.3680043.8.498.40873966949665452930715805562663419676","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934","series":"1.2.826.0.1.3680043.8.498.11641597169026035810166608388029353143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.14699563163949700199301059347317782523","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99155175553749543768821558247962844052","series":"1.2.826.0.1.3680043.8.498.12811495114632538180624790377568056249","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908","series":"1.2.826.0.1.3680043.8.498.90614750058567464680481684494676718933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99303079292320085374389086609761974820","series":"1.2.826.0.1.3680043.8.498.34579698644644707733751955518347684439","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99414163787650487720542840540379270468","series":"1.2.826.0.1.3680043.8.498.86052194365536965755158560885392329958","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99441773159476751698123795913150660766","series":"1.2.826.0.1.3680043.8.498.11899368256839738621553100899022016458","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99454657594216441037514296544803013407","series":"1.2.826.0.1.3680043.8.498.51830287579366468569087365077710221183","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961","series":"1.2.826.0.1.3680043.8.498.23500863596395323554552819393073935415","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095","series":"1.2.826.0.1.3680043.8.498.57501207546473854347283957120399484614","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99651725851820083142440965238449655973","series":"1.2.826.0.1.3680043.8.498.38135480996575725720494083119227716539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711","series":"1.2.826.0.1.3680043.8.498.10218870855492400399063837700559036137","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99813269006669714758887909212386431934","series":"1.2.826.0.1.3680043.8.498.62968597985071725213716010246868639694","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99827460918477409069820779642161448917","series":"1.2.826.0.1.3680043.8.498.82984850564206107490948370640529838446","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99880021280666757817587834481314366555","series":"1.2.826.0.1.3680043.8.498.20295166215924439531177264827446165291","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209","series":"1.2.826.0.1.3680043.8.498.31216809385545367412195846511090977787","plane":"Sagittal","fluid":0,"slot":"SAG_T1"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1777
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
